# Candidate Generation (TBM + DRfold2 + optional external seeds)

In [1]:
# ============================================================
# NOTEBOOK-2 / STAGE A — Candidate Generation (TBM + DRfold2 + optional external seeds) (ONE CELL)
# REVISI FULL v4 (FIX: overlap column 'n_segments' during join)
#
# Your error:
# ValueError: columns overlap but no suffix specified: Index(['n_segments'], dtype='object')
# Cause:
# targets_* already contains column n_segments, then we join seg_counts also named n_segments.
# Fix:
# - Always compute seg_count as 'seg_n' then set/override df['n_segments'] safely
# ============================================================

import os, sys, re, json, time, hashlib, warnings
from pathlib import Path
from typing import Dict, Any, List, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

# ----------------------------
# 0) Helpers
# ----------------------------
REQ_SUBDIRS = ["tables","meta","tbm","msa","labels_npz","residue_index"]

def sha1_json(obj: Dict[str, Any]) -> str:
    s = json.dumps(obj, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha1(s).hexdigest()[:12]

def read_json_safe(path: Path) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Missing JSON: {path}")
    txt = path.read_text(encoding="utf-8", errors="replace").strip()
    if not txt or txt[0] not in "{[":
        raise ValueError(f"Invalid JSON content at {path} (starts with {repr(txt[:80])}).")
    return json.loads(txt)

def is_artifacts_root(d: Path) -> bool:
    return d.is_dir() and all((d / s).exists() for s in REQ_SUBDIRS)

def find_artifacts_root() -> Path:
    base = Path("/kaggle/input")
    if not base.exists():
        raise FileNotFoundError("/kaggle/input not found")

    direct = base / "stanford-rna-3d-folding-part-2-dataset" / "rna3d_artifacts_v1"
    if is_artifacts_root(direct):
        return direct

    hinted = base / "stanford-rna-3d-folding-part-2-dataset"
    if hinted.exists():
        stack = [hinted]
        for _ in range(3):
            nxt = []
            for d in stack:
                if is_artifacts_root(d):
                    return d
                for p in d.iterdir():
                    if p.is_dir():
                        nxt.append(p)
            stack = nxt

    hits = []
    for d in base.iterdir():
        if not d.is_dir():
            continue
        if is_artifacts_root(d):
            hits.append(d)
            continue
        for p in d.iterdir():
            if p.is_dir() and is_artifacts_root(p):
                hits.append(p)
        for p in d.iterdir():
            if not p.is_dir():
                continue
            for q in p.iterdir():
                if q.is_dir() and is_artifacts_root(q):
                    hits.append(q)

    if not hits:
        raise FileNotFoundError(
            "Could not find artifacts root containing: "
            f"{', '.join(REQ_SUBDIRS)} under /kaggle/input."
        )
    hits.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    return hits[0]

def safe_mkdir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def normalize_seq(s: str) -> str:
    return str(s).strip().upper().replace("T", "U")

def try_import_gemmi():
    try:
        import gemmi
        return gemmi
    except Exception:
        return None

def rebase_to_art_root(path_str: str, art_root: Path) -> Path:
    if path_str is None:
        return Path("")
    s = str(path_str)
    p = Path(s)
    if s and p.exists():
        return p
    key = "rna3d_artifacts_v1"
    if key in s:
        rel = s.split(key, 1)[1].lstrip("/\\")
        return art_root / rel
    return art_root / s.lstrip("/\\")

def pick_existing(*paths: Path) -> Path:
    for p in paths:
        if p is None:
            continue
        p = Path(p)
        if str(p) and p.exists():
            return p
    return Path("")

# ----------------------------
# 1) Locate roots
# ----------------------------
ART_ROOT = find_artifacts_root()

COMP_ROOT = Path("/kaggle/input/stanford-rna-3d-folding-2")
if not COMP_ROOT.exists():
    raise FileNotFoundError(
        "Competition dataset not found at /kaggle/input/stanford-rna-3d-folding-2. Add it as input."
    )

PDB_RNA_DIR  = COMP_ROOT / "PDB_RNA"
MSA_DIR_COMP = COMP_ROOT / "MSA"

print("=== ROOTS ===")
print("ART_ROOT     :", ART_ROOT)
print("COMP_ROOT    :", COMP_ROOT)
print("PDB_RNA_DIR  :", PDB_RNA_DIR)
print("MSA_DIR_COMP :", MSA_DIR_COMP)

# ----------------------------
# 2) Load + rebase manifest paths
# ----------------------------
META_DIR = ART_ROOT / "meta"
manifest_path = META_DIR / "artifacts_manifest_stage5.json"
manifest = read_json_safe(manifest_path) if manifest_path.exists() else {"tables": {}, "tbm": {}, "msa": {}}

tbl = manifest.get("tables", {}) or {}
tbm = manifest.get("tbm", {}) or {}
msa = manifest.get("msa", {}) or {}

targets_train_p = pick_existing(rebase_to_art_root(tbl.get("targets_train_stage2",""), ART_ROOT),
                                ART_ROOT/"tables"/"targets_train_stage2.parquet")
targets_val_p   = pick_existing(rebase_to_art_root(tbl.get("targets_val_stage2",""), ART_ROOT),
                                ART_ROOT/"tables"/"targets_val_stage2.parquet")
targets_test_p  = pick_existing(rebase_to_art_root(tbl.get("targets_test_stage2",""), ART_ROOT),
                                ART_ROOT/"tables"/"targets_test_stage2.parquet")

segments_train_p = pick_existing(rebase_to_art_root(tbl.get("segments_train",""), ART_ROOT),
                                 ART_ROOT/"tables"/"segments_train.parquet")
segments_val_p   = pick_existing(rebase_to_art_root(tbl.get("segments_val",""), ART_ROOT),
                                 ART_ROOT/"tables"/"segments_val.parquet")
segments_test_p  = pick_existing(rebase_to_art_root(tbl.get("segments_test",""), ART_ROOT),
                                 ART_ROOT/"tables"/"segments_test.parquet")

template_index_p = pick_existing(rebase_to_art_root(tbm.get("template_index",""), ART_ROOT),
                                 ART_ROOT/"tbm"/"template_index.parquet")

msa_index_train_p = pick_existing(rebase_to_art_root(msa.get("msa_index_train",""), ART_ROOT),
                                  ART_ROOT/"msa"/"msa_index_train.parquet")
msa_index_val_p   = pick_existing(rebase_to_art_root(msa.get("msa_index_val",""), ART_ROOT),
                                  ART_ROOT/"msa"/"msa_index_val.parquet")

must_exist = [
    ("targets_train_stage2", targets_train_p),
    ("targets_val_stage2", targets_val_p),
    ("targets_test_stage2", targets_test_p),
    ("segments_train", segments_train_p),
    ("segments_val", segments_val_p),
    ("segments_test", segments_test_p),
    ("template_index", template_index_p),
    ("msa_index_train", msa_index_train_p),
    ("msa_index_val", msa_index_val_p),
]
missing = [(k, str(p)) for k, p in must_exist if not p.exists()]
if missing:
    msg = "\n".join([f"- {k}: {p}" for k, p in missing])
    raise FileNotFoundError("Some required artifacts files are missing:\n" + msg)

targets_train = pd.read_parquet(targets_train_p)
targets_val   = pd.read_parquet(targets_val_p)
targets_test  = pd.read_parquet(targets_test_p)

seg_train = pd.read_parquet(segments_train_p)
seg_val   = pd.read_parquet(segments_val_p)
seg_test  = pd.read_parquet(segments_test_p)

template_index = pd.read_parquet(template_index_p)

msa_index_train = pd.read_parquet(msa_index_train_p)
msa_index_val   = pd.read_parquet(msa_index_val_p)

print("\n=== LOADED ===")
print("targets_train:", targets_train.shape)
print("targets_val  :", targets_val.shape)
print("targets_test :", targets_test.shape)
print("segments_train:", seg_train.shape)
print("template_index:", template_index.shape)

msa_train_map = dict(zip(msa_index_train["target_id"].astype("string"), msa_index_train["msa_path"].astype("string")))
msa_val_map   = dict(zip(msa_index_val["target_id"].astype("string"),   msa_index_val["msa_path"].astype("string")))

# ----------------------------
# 3) Config
# ----------------------------
CFG = {
    "RUN_SPLITS": ["val"],

    "MAX_CAND_PER_TARGET": 32,
    "TBM_TOPN_IDENTICAL_SEQ": 6,
    "TBM_TOPN_SAME_CLUSTER": 6,
    "DRFOLD2_N_SAMPLES": 20,
    "EXT_TOPN": 8,

    "TBM_REQUIRE_CIF": True,
    "TBM_MAX_RESOLUTION": 5.0,
    "TBM_MIN_FRACTION_OBS": 0.5,
    "TBM_MIN_STRUCT_ADJ": 0.2,

    "TBM_EXTRACT_ONLY_SINGLE_SEGMENT": True,
    "SKIP_TBM_IF_L_GT": 4000,
}

CFG_ID = sha1_json(CFG)
RUN_DIR = safe_mkdir(Path("/kaggle/working/rna3d_run") / "candidates" / f"cfg_{CFG_ID}")
(Path(RUN_DIR) / "cfg_candidate_gen.json").write_text(json.dumps({
    "cfg": CFG,
    "cfg_id": CFG_ID,
    "art_root": str(ART_ROOT),
    "comp_root": str(COMP_ROOT),
    "utc_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
}, indent=2), encoding="utf-8")

print("\n=== CONFIG ===")
print(json.dumps(CFG, indent=2))
print("RUN_DIR:", RUN_DIR)

# ----------------------------
# 4) Prepare targets per split (FIX overlap)
# ----------------------------
def split_targets(split: str) -> pd.DataFrame:
    if split == "train":
        df = targets_train.copy()
        if "rebuild_ok" in df.columns:
            df = df[df["rebuild_ok"] == True].copy()
        seg = seg_train
        msa_map = msa_train_map
    elif split == "val":
        df = targets_val.copy()
        seg = seg_val
        msa_map = msa_val_map
    elif split == "test":
        df = targets_test.copy()
        seg = seg_test
        msa_map = msa_val_map
    else:
        raise ValueError(f"Unknown split: {split}")

    df["target_id"] = df["target_id"].astype("string")
    df["sequence"]  = df["sequence"].astype("string").map(normalize_seq)
    df["L"]         = df["L"].astype("int32")

    # Compute segment counts safely without join overlap
    seg_n = seg.groupby("target_id")["seg_idx"].count().rename("seg_n").astype("int32")

    # if df already has n_segments, overwrite from seg_n; else create it
    if "n_segments" in df.columns:
        df["n_segments"] = df["target_id"].map(seg_n).fillna(df["n_segments"]).fillna(0).astype("int32")
    else:
        df["n_segments"] = df["target_id"].map(seg_n).fillna(0).astype("int32")

    df["is_single_segment"] = (df["n_segments"] == 1)

    def _msa_path(tid: str) -> str:
        tid = str(tid)
        p = msa_map.get(tid, "")
        if p:
            pp = Path(str(p))
            if pp.exists():
                return str(pp)
        return str(MSA_DIR_COMP / f"{tid}.MSA.fasta")

    df["msa_path"] = df["target_id"].map(_msa_path).astype("string")
    return df

# ----------------------------
# 5) TBM selection + CIF extraction
# ----------------------------
gemmi = try_import_gemmi()
print("\n=== GEMMI ===")
print("[OK] gemmi available (CIF parsing enabled)." if gemmi is not None else "[WARN] gemmi not available (TBM coords disabled; TBM_REF only).")

ti = template_index.copy()
for c in ["target_id","pdb_id","auth_chain_id","chain_id","sequence"]:
    if c in ti.columns:
        ti[c] = ti[c].astype("string")

if "sequence" in ti.columns:
    ti["sequence"] = ti["sequence"].astype("string").map(normalize_seq)
    ti["seq_len"] = ti["sequence"].str.len()

if "has_cif" in ti.columns and CFG["TBM_REQUIRE_CIF"]:
    ti = ti[ti["has_cif"] == True].copy()

if "resolution" in ti.columns:
    ti["resolution"] = pd.to_numeric(ti["resolution"], errors="coerce")
    ti = ti[(ti["resolution"].isna()) | (ti["resolution"] <= float(CFG["TBM_MAX_RESOLUTION"]))].copy()

if "fraction_observed" in ti.columns:
    ti["fraction_observed"] = pd.to_numeric(ti["fraction_observed"], errors="coerce")
    ti = ti[(ti["fraction_observed"].isna()) | (ti["fraction_observed"] >= float(CFG["TBM_MIN_FRACTION_OBS"]))].copy()

if "total_structuredness_adjusted" in ti.columns:
    ti["total_structuredness_adjusted"] = pd.to_numeric(ti["total_structuredness_adjusted"], errors="coerce")
    ti = ti[(ti["total_structuredness_adjusted"].isna()) | (ti["total_structuredness_adjusted"] >= float(CFG["TBM_MIN_STRUCT_ADJ"]))].copy()

if "pdb_id" in ti.columns:
    ti["pdb_id_lc"] = ti["pdb_id"].astype("string").str.lower()
    pdb_groups = ti.groupby("pdb_id_lc", sort=False)
else:
    pdb_groups = None

seq2rows = {}
if "sequence" in ti.columns:
    for seq, sub in ti.groupby("sequence", sort=False):
        if isinstance(seq, str) and seq:
            seq2rows[seq] = sub

CLUSTER_COLS = [c for c in ["mmseqs_0.950","mmseqs_0.900","mmseqs_0.850","seq_group_id","group_id"] if c in ti.columns]

def parse_pdb_from_target_id(target_id: str) -> str:
    return str(target_id).split("_", 1)[0].lower()

def tbm_select_templates(target_id: str, sequence: str, L: int,
                         top_identical: int, top_cluster: int) -> List[Dict[str, Any]]:
    if L > int(CFG["SKIP_TBM_IF_L_GT"]):
        return []

    pdb_lc = parse_pdb_from_target_id(target_id)
    out: List[Dict[str, Any]] = []
    used = set()

    sub = seq2rows.get(sequence, None)
    if sub is not None and len(sub) > 0:
        sub2 = sub
        if "pdb_id_lc" in sub2.columns:
            sub2 = sub2[sub2["pdb_id_lc"] != pdb_lc]
        if "resolution" in sub2.columns:
            sub2 = sub2.sort_values(["resolution"], na_position="last")
        sub2 = sub2.head(int(top_identical))
        for r in sub2.itertuples(index=False):
            tkey = (getattr(r, "pdb_id", ""), getattr(r, "auth_chain_id", ""), getattr(r, "target_id", ""))
            if tkey in used:
                continue
            used.add(tkey)
            out.append({
                "source_reason": "identical_sequence",
                "template_target_id": str(getattr(r, "target_id", "")),
                "pdb_id": str(getattr(r, "pdb_id", "")),
                "auth_chain_id": str(getattr(r, "auth_chain_id", "")),
                "chain_id": str(getattr(r, "chain_id", "")),
            })

    if top_cluster > 0 and CLUSTER_COLS and pdb_groups is not None:
        try:
            cand = pdb_groups.get_group(pdb_lc)
        except Exception:
            cand = None
        if cand is not None and len(cand) > 0:
            if "resolution" in cand.columns:
                cand = cand.sort_values(["resolution"], na_position="last")
            anchor = cand.iloc[0]
            for ccol in CLUSTER_COLS:
                cid = anchor.get(ccol, None)
                if cid is None or (isinstance(cid, float) and np.isnan(cid)):
                    continue
                subc = ti[ti[ccol] == cid].copy()
                if len(subc) == 0:
                    continue
                if "pdb_id_lc" in subc.columns:
                    subc = subc[subc["pdb_id_lc"] != pdb_lc]
                if "seq_len" in subc.columns:
                    subc["len_diff"] = (subc["seq_len"].astype("int64") - int(L)).abs()
                    if "resolution" in subc.columns:
                        subc = subc.sort_values(["len_diff","resolution"], na_position="last")
                    else:
                        subc = subc.sort_values(["len_diff"], na_position="last")
                else:
                    if "resolution" in subc.columns:
                        subc = subc.sort_values(["resolution"], na_position="last")

                subc = subc.head(int(top_cluster))
                for r in subc.itertuples(index=False):
                    tkey = (getattr(r, "pdb_id", ""), getattr(r, "auth_chain_id", ""), getattr(r, "target_id", ""))
                    if tkey in used:
                        continue
                    used.add(tkey)
                    out.append({
                        "source_reason": f"cluster:{ccol}",
                        "template_target_id": str(getattr(r, "target_id", "")),
                        "pdb_id": str(getattr(r, "pdb_id", "")),
                        "auth_chain_id": str(getattr(r, "auth_chain_id", "")),
                        "chain_id": str(getattr(r, "chain_id", "")),
                    })
                break

    return out

def extract_c1_coords_from_cif(pdb_id: str, auth_chain_id: str, chain_id_fallback: str, expect_L: int) -> Optional[np.ndarray]:
    if gemmi is None:
        return None
    pdb_id = str(pdb_id).strip()
    if not pdb_id:
        return None

    cif_path = PDB_RNA_DIR / f"{pdb_id.lower()}.cif"
    if not cif_path.exists():
        cif_path = PDB_RNA_DIR / f"{pdb_id.upper()}.cif"
    if not cif_path.exists():
        return None

    try:
        st = gemmi.read_structure(str(cif_path))
        model = st[0]

        want_auth = str(auth_chain_id).strip()
        want_chain = str(chain_id_fallback).strip()

        chain = None
        if want_auth:
            for ch in model:
                if ch.name == want_auth:
                    chain = ch
                    break
        if chain is None and want_chain:
            for ch in model:
                if ch.name == want_chain:
                    chain = ch
                    break
        if chain is None:
            return None

        coords = []
        for res in chain:
            atom = res.find_atom("C1'", "*")
            if atom is None:
                atom = res.find_atom("C1*", "*")
            if atom is None:
                continue
            pos = atom.pos
            coords.append([pos.x, pos.y, pos.z])

        if len(coords) != int(expect_L):
            return None
        return np.asarray(coords, dtype=np.float32)
    except Exception:
        return None

# ----------------------------
# 6) DRfold2 + external seeds (best-effort; non-fatal)
# ----------------------------
def autodetect_drfold2() -> Dict[str, Any]:
    info = {"available": False, "repo_dir": "", "models_dir": "", "note": "", "infer_fn": None}
    base = Path("/kaggle/input")
    repo = None
    models = None
    for d in base.iterdir():
        if not d.is_dir():
            continue
        nm = d.name.lower()
        if "drfold2" in nm and "model" not in nm and repo is None:
            repo = d
        if "drfold2" in nm and "model" in nm and models is None:
            models = d
    if repo is None:
        info["note"] = "DRfold2 repo not found under /kaggle/input (TBM-only run)."
        return info

    sys.path.insert(0, str(repo))
    info["repo_dir"] = str(repo)
    if models is not None:
        info["models_dir"] = str(models)

    for mod_name, fn_name in [
        ("drfold2", "infer"),
        ("drfold2", "predict"),
        ("drfold2.inference", "predict"),
        ("drfold2.inference", "infer"),
    ]:
        try:
            m = __import__(mod_name, fromlist=[fn_name])
            fn = getattr(m, fn_name, None)
            if callable(fn):
                info["available"] = True
                info["infer_fn"] = fn
                info["note"] = f"Imported {mod_name}.{fn_name}"
                return info
        except Exception:
            pass

    info["note"] = "DRfold2 repo found but no known entrypoint imported; adapt autodetect_drfold2()."
    return info

DRF = autodetect_drfold2()
print("\n=== DRFOLD2 DETECT ===")
print(json.dumps({k: (v if k != "infer_fn" else ("<callable>" if v else None)) for k, v in DRF.items()}, indent=2))

def run_drfold2_candidates(sequence: str, msa_path: str, n_samples: int) -> List[np.ndarray]:
    if not DRF["available"] or DRF["infer_fn"] is None:
        return []
    fn = DRF["infer_fn"]
    try:
        out = fn(sequence=sequence, msa_path=msa_path, n_samples=n_samples, models_dir=DRF["models_dir"])
    except TypeError:
        try:
            out = fn(sequence, msa_path, n_samples)
        except Exception:
            return []
    except Exception:
        return []

    arrs: List[np.ndarray] = []
    if out is None:
        return arrs
    if isinstance(out, np.ndarray):
        if out.ndim == 3:
            for i in range(out.shape[0]):
                arrs.append(np.asarray(out[i], dtype=np.float32))
        elif out.ndim == 2 and out.shape[1] == 3:
            arrs.append(np.asarray(out, dtype=np.float32))
        return arrs
    if isinstance(out, (list, tuple)):
        for x in out:
            x = np.asarray(x)
            if x.ndim == 2 and x.shape[1] == 3:
                arrs.append(x.astype(np.float32, copy=False))
    return arrs

def autodetect_external_seed_dir() -> Optional[Path]:
    base = Path("/kaggle/input")
    for d in base.iterdir():
        if not d.is_dir():
            continue
        nm = d.name.lower()
        if ("boltz" in nm) or ("seed" in nm and "rna" in nm):
            if len(list(d.rglob("*.npz"))) > 0:
                return d
    return None

EXT_DIR = autodetect_external_seed_dir()
print("\n=== EXTERNAL SEEDS ===")
print(f"[OK] External seed dataset detected: {EXT_DIR}" if EXT_DIR else "[INFO] No external seed dataset detected (optional).")

def load_external_seeds(target_id: str, max_take: int) -> List[np.ndarray]:
    if EXT_DIR is None:
        return []
    tid = str(target_id)
    files = (list(EXT_DIR.rglob(f"{tid}.npz")) + list(EXT_DIR.rglob(f"{tid}_*.npz")))[:10]
    out: List[np.ndarray] = []
    for fp in files:
        try:
            z = np.load(fp, allow_pickle=True)
            if "coords" in z:
                x = np.asarray(z["coords"])
                if x.ndim == 3:
                    for i in range(min(x.shape[0], max_take - len(out))):
                        out.append(np.asarray(x[i], dtype=np.float32))
                elif x.ndim == 2 and x.shape[1] == 3:
                    out.append(np.asarray(x, dtype=np.float32))
            elif "coords_list" in z:
                for x in z["coords_list"]:
                    if len(out) >= max_take:
                        break
                    x = np.asarray(x)
                    if x.ndim == 2 and x.shape[1] == 3:
                        out.append(x.astype(np.float32, copy=False))
        except Exception:
            pass
        if len(out) >= max_take:
            break
    return out[:max_take]

# ----------------------------
# 7) Candidate generation + saving
# ----------------------------
def save_candidate_npz(path: Path, target_id: str, method: str, source: str, coords: np.ndarray):
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez(
        path,
        target_id=str(target_id),
        method=str(method),
        source=str(source),
        L=np.int32(coords.shape[0]),
        coords=coords.astype(np.float32, copy=False),
    )

def candidate_dir(split: str, target_id: str) -> Path:
    return RUN_DIR / split / str(target_id)

def run_split(split: str) -> pd.DataFrame:
    df = split_targets(split)
    n = len(df)
    rows = []

    print(f"\n=== GENERATE CANDIDATES: {split} (n_targets={n:,}) ===")

    for i, r in enumerate(df.itertuples(index=False), 1):
        tid = str(r.target_id)
        seq = str(r.sequence)
        L   = int(r.L)
        msa_path = str(r.msa_path)
        is_single = bool(r.is_single_segment)

        recorded = 0
        cdir = candidate_dir(split, tid)

        templ = tbm_select_templates(
            target_id=tid, sequence=seq, L=L,
            top_identical=int(CFG["TBM_TOPN_IDENTICAL_SEQ"]),
            top_cluster=int(CFG["TBM_TOPN_SAME_CLUSTER"]),
        )

        do_extract = (gemmi is not None) and (not CFG["TBM_EXTRACT_ONLY_SINGLE_SEGMENT"] or is_single)

        tbm_written = 0
        tbm_ref_written = 0
        for t in templ:
            if recorded >= int(CFG["MAX_CAND_PER_TARGET"]):
                break
            source = f"{t['source_reason']}|{t['pdb_id']}|auth={t.get('auth_chain_id','')}"
            coords = None
            if do_extract:
                coords = extract_c1_coords_from_cif(
                    pdb_id=t["pdb_id"],
                    auth_chain_id=t.get("auth_chain_id",""),
                    chain_id_fallback=t.get("chain_id",""),
                    expect_L=L,
                )

            if coords is None:
                tbm_ref_written += 1
                rows.append({
                    "split": split, "target_id": tid, "L": L,
                    "cand_id": f"tbm_ref_{tbm_ref_written:03d}",
                    "method": "TBM_REF",
                    "source": source,
                    "coords_path": "",
                    "msa_path": msa_path,
                })
                recorded += 1
                continue

            tbm_written += 1
            fpath = cdir / f"tbm_{tbm_written:03d}.npz"
            save_candidate_npz(fpath, tid, "TBM", source, coords)
            rows.append({
                "split": split, "target_id": tid, "L": L,
                "cand_id": fpath.stem,
                "method": "TBM",
                "source": source,
                "coords_path": str(fpath),
                "msa_path": msa_path,
            })
            recorded += 1

        if recorded < int(CFG["MAX_CAND_PER_TARGET"]):
            ext = load_external_seeds(tid, max_take=int(CFG["EXT_TOPN"]))
            ext_written = 0
            for coords in ext:
                if recorded >= int(CFG["MAX_CAND_PER_TARGET"]):
                    break
                if not (isinstance(coords, np.ndarray) and coords.shape == (L,3)):
                    continue
                ext_written += 1
                fpath = cdir / f"ext_{ext_written:03d}.npz"
                save_candidate_npz(fpath, tid, "EXT", f"external:{EXT_DIR.name if EXT_DIR else ''}", coords)
                rows.append({
                    "split": split, "target_id": tid, "L": L,
                    "cand_id": fpath.stem,
                    "method": "EXT",
                    "source": f"external:{EXT_DIR.name if EXT_DIR else ''}",
                    "coords_path": str(fpath),
                    "msa_path": msa_path,
                })
                recorded += 1

        if recorded < int(CFG["MAX_CAND_PER_TARGET"]):
            drs = run_drfold2_candidates(seq, msa_path, n_samples=int(CFG["DRFOLD2_N_SAMPLES"]))
            dr_written = 0
            for coords in drs:
                if recorded >= int(CFG["MAX_CAND_PER_TARGET"]):
                    break
                if not (isinstance(coords, np.ndarray) and coords.shape == (L,3)):
                    continue
                dr_written += 1
                fpath = cdir / f"drfold2_{dr_written:03d}.npz"
                save_candidate_npz(fpath, tid, "DRFOLD2", DRF.get("note","drfold2"), coords)
                rows.append({
                    "split": split, "target_id": tid, "L": L,
                    "cand_id": fpath.stem,
                    "method": "DRFOLD2",
                    "source": DRF.get("note","drfold2"),
                    "coords_path": str(fpath),
                    "msa_path": msa_path,
                })
                recorded += 1

        if recorded == 0:
            rows.append({
                "split": split, "target_id": tid, "L": L,
                "cand_id": "none_000",
                "method": "NONE",
                "source": "no_candidates_generated",
                "coords_path": "",
                "msa_path": msa_path,
            })

        if i % 25 == 0 or i == n:
            n_coords = sum(1 for x in rows if x["split"] == split and x["coords_path"])
            print(f"[{split}] {i:,}/{n:,} | candidates_with_coords={n_coords:,}", end="\r")

    print()
    man_df = pd.DataFrame(rows)
    man_path = RUN_DIR / f"manifest_candidates_{split}.parquet"
    man_df.to_parquet(man_path, index=False)

    n_total = len(man_df)
    n_coords = int((man_df["coords_path"].astype("string").str.len() > 0).sum())
    n_tbmref = int((man_df["method"] == "TBM_REF").sum())
    print(f"\n[{split}] manifest rows={n_total:,} | with_coords={n_coords:,} | TBM_REF(no coords)={n_tbmref:,}")
    print(f"[{split}] saved: {man_path}")
    return man_df

# ----------------------------
# 8) Run
# ----------------------------
all_manifests = {}
for sp in CFG["RUN_SPLITS"]:
    all_manifests[sp] = run_split(sp)

print("\n[OK] Candidate generation complete.")
print("RUN_DIR:", RUN_DIR)
print("Next stage: Accurate Scoring + Robust Ranking (select top-K) using manifest_candidates_*.parquet")


=== ROOTS ===
ART_ROOT     : /kaggle/input/stanford-rna-3d-folding-part-2-dataset/rna3d_artifacts_v1
COMP_ROOT    : /kaggle/input/stanford-rna-3d-folding-2
PDB_RNA_DIR  : /kaggle/input/stanford-rna-3d-folding-2/PDB_RNA
MSA_DIR_COMP : /kaggle/input/stanford-rna-3d-folding-2/MSA

=== LOADED ===
targets_train: (5716, 19)
targets_val  : (28, 19)
targets_test : (28, 19)
segments_train: (17015, 9)
template_index: (26255, 35)

=== CONFIG ===
{
  "RUN_SPLITS": [
    "val"
  ],
  "MAX_CAND_PER_TARGET": 32,
  "TBM_TOPN_IDENTICAL_SEQ": 6,
  "TBM_TOPN_SAME_CLUSTER": 6,
  "DRFOLD2_N_SAMPLES": 20,
  "EXT_TOPN": 8,
  "TBM_REQUIRE_CIF": true,
  "TBM_MAX_RESOLUTION": 5.0,
  "TBM_MIN_FRACTION_OBS": 0.5,
  "TBM_MIN_STRUCT_ADJ": 0.2,
  "TBM_EXTRACT_ONLY_SINGLE_SEGMENT": true,
  "SKIP_TBM_IF_L_GT": 4000
}
RUN_DIR: /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac

=== GEMMI ===
[WARN] gemmi not available (TBM coords disabled; TBM_REF only).

=== DRFOLD2 DETECT ===
{
  "available": false,
  "repo_dir": 

# Standardize & Validate Candidates (geometry + indexing + precision)

In [2]:
# ============================================================
# NOTEBOOK-2 / STAGE B — Standardize & Validate Candidates
# (geometry + indexing + precision) (ONE CELL)
#
# What it does (EN):
# - Load candidate manifest(s) from RUN_DIR
# - Validate: shape/L match, finite coords, basic geometry sanity
# - Validate indexing coverage using residue_index (optional but enabled for val/test)
# - Standardize: write standardized NPZ (float64 by default for ranking precision)
# - Export: standardized_manifest_*.parquet + QA csv summary
#
# Penjelasan (ID):
# - Tahap ini memastikan setiap kandidat koordinat "rapi" dan aman dipakai tahap scoring/ranking.
# - Kalau kandidat belum punya coords (TBM_REF saja), tahap ini tidak crash: tetap buat QA & manifest.
# - Float64 disiapkan supaya tahap ranking lebih stabil (meniru insight 1st place: double precision).
# ============================================================

import os, json, time, math, warnings, hashlib
from pathlib import Path
from typing import Dict, Any, Optional, Tuple, List

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

# ----------------------------
# 0) Utilities (safe I/O)
# ----------------------------
def read_json_safe(path: Path) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Missing JSON: {path}")
    txt = path.read_text(encoding="utf-8", errors="replace").strip()
    # Prevent "Unexpected token '<' ..." / HTML garbage
    if not txt or txt[0] not in "{[":
        raise ValueError(f"Invalid JSON content at {path} (starts with {repr(txt[:80])}).")
    return json.loads(txt)

def safe_mkdir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def is_finite_array(x: np.ndarray) -> bool:
    return np.isfinite(x).all()

def load_npz_coords(path: Path) -> Optional[np.ndarray]:
    try:
        z = np.load(path, allow_pickle=False)
        if "coords" in z:
            c = np.asarray(z["coords"])
            return c
        return None
    except Exception:
        return None

def basic_geometry_checks(coords: np.ndarray) -> Dict[str, Any]:
    """
    Lightweight geometry sanity (O(L)):
    - no all-zeros / near-constant
    - consecutive distance stats within loose bounds
    - no extreme coordinate magnitude (not strict; later scoring may clip for submission)
    """
    out = {
        "finite_ok": False,
        "var_ok": False,
        "bond_ok": False,
        "bond_mean": np.nan,
        "bond_p01": np.nan,
        "bond_p99": np.nan,
        "bond_bad_frac": np.nan,
        "coord_absmax": np.nan,
        "geom_ok": False,
    }

    c = np.asarray(coords)
    if c.ndim != 2 or c.shape[1] != 3:
        return out
    if not is_finite_array(c):
        return out
    out["finite_ok"] = True

    # variance / collapse check
    # (use float64 for stable stats)
    c64 = c.astype(np.float64, copy=False)
    std = float(np.std(c64))
    out["var_ok"] = (std > 1e-6)

    # consecutive "bond" distances (C1' approx, loose)
    if c64.shape[0] >= 2:
        d = np.linalg.norm(c64[1:] - c64[:-1], axis=1)
        out["bond_mean"] = float(np.mean(d))
        out["bond_p01"]  = float(np.quantile(d, 0.01))
        out["bond_p99"]  = float(np.quantile(d, 0.99))

        # loose bounds to catch broken structures
        bad = (d < 0.5) | (d > 20.0) | ~np.isfinite(d)
        out["bond_bad_frac"] = float(np.mean(bad))
        out["bond_ok"] = (out["bond_bad_frac"] <= 0.20)  # allow some weirdness
    else:
        out["bond_ok"] = True

    out["coord_absmax"] = float(np.max(np.abs(c64))) if c64.size else np.nan

    # overall geometry ok
    out["geom_ok"] = bool(out["finite_ok"] and out["var_ok"] and out["bond_ok"])
    return out

def load_residue_index_counts(residue_index_dir: Path, split: str) -> pd.Series:
    """
    Returns Series: target_id -> residue_count
    This confirms indexing coverage/length for each target.
    """
    part_glob = residue_index_dir / split / "part-*.parquet"
    files = sorted(part_glob.parent.glob(part_glob.name))
    if not files:
        return pd.Series(dtype="int64")
    # For val/test it's tiny; for train could be many parts (avoid here).
    df_list = []
    for f in files:
        dfp = pd.read_parquet(f, columns=["target_id"])
        df_list.append(dfp)
    df = pd.concat(df_list, ignore_index=True)
    return df.groupby("target_id").size()

# ----------------------------
# 1) Locate RUN_DIR (from previous Stage A output)
# ----------------------------
# If RUN_DIR is already defined in the notebook, use it; else try best-effort auto-detect.
if "RUN_DIR" in globals():
    RUN_DIR = Path(RUN_DIR)
else:
    base = Path("/kaggle/working/rna3d_run/candidates")
    if not base.exists():
        raise FileNotFoundError("RUN_DIR not found. Please run Candidate Generation stage first.")
    cfg_dirs = sorted([d for d in base.iterdir() if d.is_dir() and d.name.startswith("cfg_")],
                      key=lambda x: x.stat().st_mtime, reverse=True)
    if not cfg_dirs:
        raise FileNotFoundError("No cfg_* folder found under /kaggle/working/rna3d_run/candidates.")
    RUN_DIR = cfg_dirs[0]

cfg_path = RUN_DIR / "cfg_candidate_gen.json"
CFG_WRAP = read_json_safe(cfg_path)
CFG = CFG_WRAP.get("cfg", {})
ART_ROOT = Path(CFG_WRAP.get("art_root", ""))
COMP_ROOT = Path(CFG_WRAP.get("comp_root", ""))

print("=== STAGE B INPUTS ===")
print("RUN_DIR  :", RUN_DIR)
print("CFG_PATH :", cfg_path)
print("ART_ROOT :", ART_ROOT)
print("COMP_ROOT:", COMP_ROOT)

# Validate minimal
if not ART_ROOT.exists():
    raise FileNotFoundError(f"ART_ROOT missing: {ART_ROOT}")
if not (ART_ROOT / "residue_index").exists():
    raise FileNotFoundError(f"residue_index not found in ART_ROOT: {ART_ROOT/'residue_index'}")

# ----------------------------
# 2) Load target tables (for L reference)
# ----------------------------
targets_train_p = ART_ROOT / "tables" / "targets_train_stage2.parquet"
targets_val_p   = ART_ROOT / "tables" / "targets_val_stage2.parquet"
targets_test_p  = ART_ROOT / "tables" / "targets_test_stage2.parquet"

targets_train = pd.read_parquet(targets_train_p)
targets_val   = pd.read_parquet(targets_val_p)
targets_test  = pd.read_parquet(targets_test_p)

targets_map = {
    "train": targets_train.set_index("target_id"),
    "val":   targets_val.set_index("target_id"),
    "test":  targets_test.set_index("target_id"),
}

# ----------------------------
# 3) Residue-index counts (indexing validation)
# ----------------------------
RESIDX_DIR = ART_ROOT / "residue_index"
# Only load counts for splits we will process (val/test by default in Candidate Gen)
splits = list(CFG.get("RUN_SPLITS", ["val"]))
res_counts = {}
for sp in splits:
    # val/test is small; train is huge (avoid in this stage unless you really want).
    if sp in ["val", "test"]:
        res_counts[sp] = load_residue_index_counts(RESIDX_DIR, sp)
    else:
        res_counts[sp] = pd.Series(dtype="int64")

# ----------------------------
# 4) Standardize settings
# ----------------------------
STD_CFG = {
    "WRITE_FLOAT64": True,     # keep float64 NPZ for stable ranking later
    "CLIP_FOR_STD": False,     # do NOT clip here (clip later only for submission export)
    "CLIP_MIN": -999.999,
    "CLIP_MAX": 9999.999,
}
STD_ID = hashlib.sha1(json.dumps(STD_CFG, sort_keys=True).encode()).hexdigest()[:10]
STD_DIR = safe_mkdir(RUN_DIR / "standardized" / f"std_{STD_ID}")
(Path(STD_DIR) / "cfg_standardize.json").write_text(json.dumps({
    "std_cfg": STD_CFG,
    "std_id": STD_ID,
    "utc_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
}, indent=2), encoding="utf-8")

print("\n=== STANDARDIZE CFG ===")
print(json.dumps(STD_CFG, indent=2))
print("STD_DIR:", STD_DIR)

# ----------------------------
# 5) Process each split
# ----------------------------
def process_split(sp: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    man_path = RUN_DIR / f"manifest_candidates_{sp}.parquet"
    if not man_path.exists():
        raise FileNotFoundError(f"Missing manifest: {man_path}")

    man = pd.read_parquet(man_path)
    man["target_id"] = man["target_id"].astype("string")
    man["method"] = man["method"].astype("string")
    man["cand_id"] = man["cand_id"].astype("string")
    man["coords_path"] = man["coords_path"].astype("string")

    # reference L from targets table
    tmap = targets_map[sp]
    if "L" in tmap.columns:
        Lref = tmap["L"]
    else:
        raise ValueError(f"targets_{sp} missing column 'L'")

    # residue index counts (optional)
    rc = res_counts.get(sp, pd.Series(dtype="int64"))

    rows = []
    qa_rows = []

    n_total = len(man)
    n_with_coords = int((man["coords_path"].str.len() > 0).sum())

    print(f"\n=== STAGE B: {sp} ===")
    print(f"manifest rows={n_total:,} | with_coords={n_with_coords:,}")

    for i, r in enumerate(man.itertuples(index=False), 1):
        tid = str(r.target_id)
        cid = str(r.cand_id)
        method = str(r.method)
        src = str(getattr(r, "source", ""))

        L_expected = int(Lref.get(tid, -1))
        idx_count = int(rc.get(tid, -1)) if tid in rc.index else -1
        index_ok = (idx_count == -1) or (idx_count == L_expected)  # if no rc, don't fail

        coords_in_path = str(r.coords_path)
        coords_exists = bool(coords_in_path) and Path(coords_in_path).exists()

        shape_ok = False
        L_ok = False
        finite_ok = False
        geom_ok = False

        std_path = ""

        geom_stats = {
            "bond_mean": np.nan,
            "bond_p01": np.nan,
            "bond_p99": np.nan,
            "bond_bad_frac": np.nan,
            "coord_absmax": np.nan,
        }

        if coords_exists:
            coords = load_npz_coords(Path(coords_in_path))
            if coords is not None and isinstance(coords, np.ndarray):
                shape_ok = (coords.ndim == 2 and coords.shape[1] == 3)
                if shape_ok:
                    L_ok = (int(coords.shape[0]) == L_expected)

                    # geometry checks in float64 (precision)
                    g = basic_geometry_checks(coords)
                    finite_ok = bool(g["finite_ok"])
                    geom_ok = bool(g["geom_ok"])
                    for k in geom_stats:
                        geom_stats[k] = g.get(k, np.nan)

                    # write standardized coords if ok shape+L and finite
                    if shape_ok and L_ok and finite_ok:
                        c = coords.astype(np.float64 if STD_CFG["WRITE_FLOAT64"] else np.float32, copy=False)
                        if STD_CFG["CLIP_FOR_STD"]:
                            c = np.clip(c, STD_CFG["CLIP_MIN"], STD_CFG["CLIP_MAX"])

                        out_dir = safe_mkdir(STD_DIR / sp / tid)
                        out_npz = out_dir / f"{cid}.npz"
                        np.savez(
                            out_npz,
                            target_id=tid,
                            cand_id=cid,
                            method=method,
                            source=src,
                            L=np.int32(L_expected),
                            coords=c,
                        )
                        std_path = str(out_npz)

        valid_ok = bool(index_ok and (not coords_exists or (shape_ok and L_ok and finite_ok)))

        rows.append({
            "split": sp,
            "target_id": tid,
            "cand_id": cid,
            "method": method,
            "source": src,
            "L_expected": L_expected,
            "index_count": idx_count,
            "index_ok": bool(index_ok),
            "coords_path": coords_in_path,
            "coords_exists": bool(coords_exists),
            "shape_ok": bool(shape_ok),
            "L_ok": bool(L_ok),
            "finite_ok": bool(finite_ok),
            "geom_ok": bool(geom_ok),
            "std_coords_path": std_path,
            "valid_ok": bool(valid_ok),
            **geom_stats,
            "msa_path": str(getattr(r, "msa_path", "")),
        })

        if i % 50 == 0 or i == n_total:
            print(f"[{sp}] processed {i:,}/{n_total:,}", end="\r")

    print()
    df_out = pd.DataFrame(rows)

    # QA summary
    qa = {
        "split": sp,
        "manifest_rows": int(n_total),
        "with_coords": int(n_with_coords),
        "coords_exists": int(df_out["coords_exists"].sum()),
        "std_written": int((df_out["std_coords_path"].astype("string").str.len() > 0).sum()),
        "index_ok": int(df_out["index_ok"].sum()),
        "shape_ok": int(df_out["shape_ok"].sum()),
        "L_ok": int(df_out["L_ok"].sum()),
        "finite_ok": int(df_out["finite_ok"].sum()),
        "geom_ok": int(df_out["geom_ok"].sum()),
        "valid_ok": int(df_out["valid_ok"].sum()),
    }
    qa_rows.append(qa)

    qa_df = pd.DataFrame(qa_rows)

    # Save outputs
    out_manifest_path = STD_DIR / f"standardized_manifest_{sp}.parquet"
    out_qa_path = STD_DIR / f"qa_stageB_{sp}.csv"
    df_out.to_parquet(out_manifest_path, index=False)
    qa_df.to_csv(out_qa_path, index=False)

    print(f"[{sp}] saved standardized manifest: {out_manifest_path}")
    print(f"[{sp}] saved QA: {out_qa_path}")
    print(f"[{sp}] QA:", qa)

    return df_out, qa_df

all_std = {}
all_qa = []
for sp in splits:
    df_std, df_qa = process_split(sp)
    all_std[sp] = df_std
    all_qa.append(df_qa)

qa_all = pd.concat(all_qa, ignore_index=True) if all_qa else pd.DataFrame()
qa_all_path = STD_DIR / "qa_stageB_ALL.csv"
qa_all.to_csv(qa_all_path, index=False)

print("\n[OK] STAGE B complete.")
print("STD_DIR:", STD_DIR)
print("QA_ALL :", qa_all_path)

# Extra note (ID):
# Kalau std_written=0 seperti kasus kamu sekarang, itu normal karena kandidat belum punya coords.
# Solusi: tambahkan DRfold2 repo+models atau external seeds dataset, atau aktifkan gemmi wheels.


=== STAGE B INPUTS ===
RUN_DIR  : /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac
CFG_PATH : /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/cfg_candidate_gen.json
ART_ROOT : /kaggle/input/stanford-rna-3d-folding-part-2-dataset/rna3d_artifacts_v1
COMP_ROOT: /kaggle/input/stanford-rna-3d-folding-2

=== STANDARDIZE CFG ===
{
  "WRITE_FLOAT64": true,
  "CLIP_FOR_STD": false,
  "CLIP_MIN": -999.999,
  "CLIP_MAX": 9999.999
}
STD_DIR: /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/standardized/std_3f9797574f

=== STAGE B: val ===
manifest rows=51 | with_coords=0
[val] processed 51/51
[val] saved standardized manifest: /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/standardized/std_3f9797574f/standardized_manifest_val.parquet
[val] saved QA: /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/standardized/std_3f9797574f/qa_stageB_val.csv
[val] QA: {'split': 'val', 'manifest_rows': 51, 'with_coords': 0, 'coords_exists': 0, 'std_written': 0, 'index_ok': 51, 'shape_ok

# Accurate Scoring + Robust Ranking (select top-K reliably)

In [3]:
# ============================================================
# NOTEBOOK-2 / STAGE C — Accurate Scoring + Robust Ranking (select top-K reliably) (ONE CELL)
#
# EN:
# - Load standardized_manifest_{split}.parquet from STAGE B
# - If coords exist: compute robust float64 geometry score (bond/curvature/clash/rg) with torch
# - Deterministic pair sampling per (target_id,cand_id) for stable ranking
# - Rank per target, select TOPK (default 5), export artifacts
#
# ID:
# - Kalau ada coords: dihitung skor geometri (float64) lalu diranking per target.
# - Kalau belum ada coords (seperti kasus kamu sekarang): tetap buat ranking fallback agar pipeline lanjut.
# - Output utama: topk_{split}.parquet + scores_{split}.parquet + QA json
# ============================================================

import os, json, time, math, hashlib, warnings
from pathlib import Path
from typing import Dict, Any, Tuple, List, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

# ----------------------------
# 0) Safe helpers
# ----------------------------
def read_json_safe(path: Path) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Missing JSON: {path}")
    txt = path.read_text(encoding="utf-8", errors="replace").strip()
    # Avoid "Unexpected token '<' ..."
    if not txt or txt[0] not in "{[":
        raise ValueError(f"Invalid JSON content at {path} (starts with {repr(txt[:80])}).")
    return json.loads(txt)

def safe_mkdir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def load_npz_coords(path: Path) -> Optional[np.ndarray]:
    try:
        z = np.load(path, allow_pickle=False)
        if "coords" in z:
            c = np.asarray(z["coords"])
            return c
        return None
    except Exception:
        return None

def stable_hash_int(s: str) -> int:
    return int(hashlib.sha1(s.encode("utf-8")).hexdigest()[:8], 16)

# ----------------------------
# 1) Locate STD_DIR and splits
# ----------------------------
if "STD_DIR" in globals():
    STD_DIR = Path(STD_DIR)
else:
    # best-effort autodetect from latest standardized folder under last RUN_DIR
    if "RUN_DIR" in globals():
        RUN_DIR = Path(RUN_DIR)
    else:
        base = Path("/kaggle/working/rna3d_run/candidates")
        if not base.exists():
            raise FileNotFoundError("No /kaggle/working/rna3d_run/candidates found. Run Stage A/B first.")
        cfg_dirs = sorted([d for d in base.iterdir() if d.is_dir() and d.name.startswith("cfg_")],
                          key=lambda x: x.stat().st_mtime, reverse=True)
        if not cfg_dirs:
            raise FileNotFoundError("No cfg_* folder found. Run Stage A/B first.")
        RUN_DIR = cfg_dirs[0]
    std_root = RUN_DIR / "standardized"
    if not std_root.exists():
        raise FileNotFoundError(f"standardized folder not found: {std_root}")
    std_dirs = sorted([d for d in std_root.iterdir() if d.is_dir() and d.name.startswith("std_")],
                      key=lambda x: x.stat().st_mtime, reverse=True)
    if not std_dirs:
        raise FileNotFoundError(f"No std_* folder found under: {std_root}")
    STD_DIR = std_dirs[0]

cfg_std_path = STD_DIR / "cfg_standardize.json"
STD_WRAP = read_json_safe(cfg_std_path)
STD_CFG = STD_WRAP.get("std_cfg", {})

# Candidate-gen cfg (for RUN_SPLITS reference)
RUN_DIR = STD_DIR.parent.parent  # .../cfg_xxx
cfg_cand_path = RUN_DIR / "cfg_candidate_gen.json"
CAND_WRAP = read_json_safe(cfg_cand_path)
CFG = CAND_WRAP.get("cfg", {})
splits = list(CFG.get("RUN_SPLITS", ["val"]))

print("=== STAGE C INPUTS ===")
print("RUN_DIR :", RUN_DIR)
print("STD_DIR :", STD_DIR)
print("splits  :", splits)
print("STD_CFG :", STD_CFG)

OUT_DIR = safe_mkdir(RUN_DIR / "ranking" / f"rank_{hashlib.sha1(json.dumps({'v':1,'std':STD_CFG}, sort_keys=True).encode()).hexdigest()[:10]}")
(Path(OUT_DIR) / "cfg_stageC.json").write_text(json.dumps({
    "stage": "C",
    "utc_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "std_dir": str(STD_DIR),
    "run_dir": str(RUN_DIR),
    "splits": splits,
    "topk": 5,
}, indent=2), encoding="utf-8")

print("OUT_DIR:", OUT_DIR)

# ----------------------------
# 2) Torch setup (optional but recommended)
# ----------------------------
try:
    import torch
    _HAS_TORCH = True
except Exception:
    _HAS_TORCH = False

device = None
if _HAS_TORCH:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.set_grad_enabled(False)

print("TORCH:", _HAS_TORCH, "| device:", str(device) if device is not None else "none")

# ----------------------------
# 3) Scoring function (float64, robust + deterministic sampling)
# ----------------------------
SCORE_CFG = {
    "TOPK": 5,
    "BOND_TARGET": 6.0,         # loose target for consecutive C1' distance (Å)
    "BOND_W": 1.0,
    "CURV_W": 0.05,
    "CLASH_THRESH": 2.0,        # Å
    "CLASH_W": 10.0,
    "RG_W": 0.2,
    "PAIR_SAMPLES": 20000,      # sample long-range pairs to approximate clashes
    "MIN_SEQ_SEP": 4,           # ignore near neighbors for clash
}

def score_coords_float64(coords_np: np.ndarray, seed: int) -> Tuple[float, Dict[str, float]]:
    """
    Lower score is better.
    Components:
    - bond: mean squared deviation of consecutive distances from target
    - curv: mean squared second-difference magnitude (smoothness)
    - clash: sampled long-range pair penalties for distances < thresh
    - rg: radius-of-gyration penalty vs sqrt(L) scaling
    """
    c = np.asarray(coords_np)
    L = int(c.shape[0])
    if (c.ndim != 2) or (c.shape[1] != 3) or (L <= 1) or (not np.isfinite(c).all()):
        return float("inf"), {"bond": np.nan, "curv": np.nan, "clash": np.nan, "rg": np.nan}

    # Use torch float64 if available, else numpy float64
    if _HAS_TORCH:
        tc = torch.as_tensor(c, dtype=torch.float64, device=device)

        # bond
        d = torch.linalg.norm(tc[1:] - tc[:-1], dim=1)
        bond = torch.mean((d - SCORE_CFG["BOND_TARGET"]) ** 2)

        # curvature
        if L >= 3:
            dd = tc[2:] - 2.0 * tc[1:-1] + tc[:-2]
            curv = torch.mean(torch.sum(dd * dd, dim=1))
        else:
            curv = torch.tensor(0.0, dtype=torch.float64, device=device)

        # rg
        center = torch.mean(tc, dim=0, keepdim=True)
        rg = torch.sqrt(torch.mean(torch.sum((tc - center) ** 2, dim=1)) + 1e-12)
        rg_target = 2.0 * math.sqrt(max(L, 1))  # loose scaling
        rg_pen = ((rg / rg_target) - 1.0) ** 2

        # clash sampling
        # deterministic
        g = torch.Generator(device="cpu")
        g.manual_seed(int(seed) & 0x7FFFFFFF)

        # sample i uniformly, sample j with min seq sep
        M = int(SCORE_CFG["PAIR_SAMPLES"])
        if L <= SCORE_CFG["MIN_SEQ_SEP"] + 1:
            clash = torch.tensor(0.0, dtype=torch.float64, device=device)
        else:
            i = torch.randint(low=0, high=L - SCORE_CFG["MIN_SEQ_SEP"], size=(M,), generator=g, device="cpu")
            # choose offset >= MIN_SEQ_SEP
            off = torch.randint(low=SCORE_CFG["MIN_SEQ_SEP"], high=L, size=(M,), generator=g, device="cpu")
            j = (i + off) % L
            i = i.to(device)
            j = j.to(device)
            pi = tc[i]
            pj = tc[j]
            dij = torch.linalg.norm(pi - pj, dim=1)
            # penalty for too-close
            thr = SCORE_CFG["CLASH_THRESH"]
            clash = torch.mean(torch.clamp(thr - dij, min=0.0) ** 2)

        total = (
            SCORE_CFG["BOND_W"] * bond
            + SCORE_CFG["CURV_W"] * curv
            + SCORE_CFG["CLASH_W"] * clash
            + SCORE_CFG["RG_W"] * rg_pen
        )

        comp = {
            "bond": float(bond.detach().cpu().numpy()),
            "curv": float(curv.detach().cpu().numpy()),
            "clash": float(clash.detach().cpu().numpy()),
            "rg": float(rg_pen.detach().cpu().numpy()),
        }
        return float(total.detach().cpu().numpy()), comp

    else:
        c64 = c.astype(np.float64, copy=False)
        d = np.linalg.norm(c64[1:] - c64[:-1], axis=1)
        bond = float(np.mean((d - SCORE_CFG["BOND_TARGET"]) ** 2))

        if L >= 3:
            dd = c64[2:] - 2.0 * c64[1:-1] + c64[:-2]
            curv = float(np.mean(np.sum(dd * dd, axis=1)))
        else:
            curv = 0.0

        center = np.mean(c64, axis=0, keepdims=True)
        rg = float(np.sqrt(np.mean(np.sum((c64 - center) ** 2, axis=1)) + 1e-12))
        rg_target = 2.0 * math.sqrt(max(L, 1))
        rg_pen = float(((rg / rg_target) - 1.0) ** 2)

        rng = np.random.default_rng(seed)
        M = int(SCORE_CFG["PAIR_SAMPLES"])
        if L <= SCORE_CFG["MIN_SEQ_SEP"] + 1:
            clash = 0.0
        else:
            i = rng.integers(0, L - SCORE_CFG["MIN_SEQ_SEP"], size=M)
            off = rng.integers(SCORE_CFG["MIN_SEQ_SEP"], L, size=M)
            j = (i + off) % L
            dij = np.linalg.norm(c64[i] - c64[j], axis=1)
            thr = SCORE_CFG["CLASH_THRESH"]
            clash = float(np.mean(np.clip(thr - dij, 0.0, None) ** 2))

        total = SCORE_CFG["BOND_W"]*bond + SCORE_CFG["CURV_W"]*curv + SCORE_CFG["CLASH_W"]*clash + SCORE_CFG["RG_W"]*rg_pen
        comp = {"bond": bond, "curv": curv, "clash": clash, "rg": rg_pen}
        return float(total), comp

# ----------------------------
# 4) Rank per split
# ----------------------------
def method_priority(method: str) -> int:
    # Lower is better (preferred)
    m = str(method).upper()
    if m == "EXT": return 0
    if m == "DRFOLD2": return 1
    if m == "TBM": return 2
    if m == "TBM_REF": return 3
    return 9

qa_rows = []
for sp in splits:
    std_manifest_path = STD_DIR / f"standardized_manifest_{sp}.parquet"
    if not std_manifest_path.exists():
        raise FileNotFoundError(f"Missing standardized manifest for split={sp}: {std_manifest_path}")

    df = pd.read_parquet(std_manifest_path)
    df["target_id"] = df["target_id"].astype("string")
    df["cand_id"] = df["cand_id"].astype("string")
    df["method"] = df["method"].astype("string")
    df["std_coords_path"] = df["std_coords_path"].astype("string")

    # Score candidates that have coords
    scores = []
    n = len(df)
    n_scored = 0

    print(f"\n=== STAGE C: scoring split={sp} rows={n:,} ===")

    for i, r in enumerate(df.itertuples(index=False), 1):
        tid = str(r.target_id)
        cid = str(r.cand_id)
        method = str(r.method)
        p = str(r.std_coords_path)

        if p and Path(p).exists():
            coords = load_npz_coords(Path(p))
            if coords is not None and coords.ndim == 2 and coords.shape[1] == 3:
                seed = stable_hash_int(f"{tid}|{cid}|{sp}")
                s, comp = score_coords_float64(coords, seed=seed)
                n_scored += 1
            else:
                s, comp = float("inf"), {"bond": np.nan, "curv": np.nan, "clash": np.nan, "rg": np.nan}
        else:
            # no coords -> fallback later
            s, comp = float("inf"), {"bond": np.nan, "curv": np.nan, "clash": np.nan, "rg": np.nan}

        scores.append((s, comp["bond"], comp["curv"], comp["clash"], comp["rg"]))

        if i % 100 == 0 or i == n:
            print(f"[{sp}] scored {i:,}/{n:,} | with_coords_scored={n_scored:,}", end="\r")

    print()
    df["score"] = [x[0] for x in scores]
    df["score_bond"] = [x[1] for x in scores]
    df["score_curv"] = [x[2] for x in scores]
    df["score_clash"] = [x[3] for x in scores]
    df["score_rg"] = [x[4] for x in scores]

    # Robust ranking per target:
    # primary: score (float64), secondary: method priority, tertiary: cand_id (stable)
    df["method_pri"] = df["method"].map(method_priority).astype("int32")

    topk_rows = []
    for tid, sub in df.groupby("target_id", sort=False):
        sub = sub.copy()

        # If everything is inf (no coords), fallback to method priority only
        all_inf = bool(np.isinf(sub["score"].to_numpy()).all())
        if all_inf:
            sub = sub.sort_values(["method_pri", "cand_id"], ascending=[True, True])
        else:
            sub = sub.sort_values(["score", "method_pri", "cand_id"], ascending=[True, True, True])

        sub_top = sub.head(int(SCORE_CFG["TOPK"])).copy()
        sub_top["rank"] = np.arange(1, len(sub_top) + 1, dtype=np.int32)

        topk_rows.append(sub_top)

    df_topk = pd.concat(topk_rows, ignore_index=True) if topk_rows else df.head(0).copy()

    # Save outputs
    scores_path = OUT_DIR / f"scores_{sp}.parquet"
    topk_path = OUT_DIR / f"topk_{sp}.parquet"
    df.to_parquet(scores_path, index=False)
    df_topk.to_parquet(topk_path, index=False)

    qa = {
        "split": sp,
        "rows": int(len(df)),
        "with_std_coords": int((df["std_coords_path"].str.len() > 0).sum()),
        "scored_coords": int(n_scored),
        "targets": int(df["target_id"].nunique()),
        "topk_rows": int(len(df_topk)),
        "topk_per_target": int(SCORE_CFG["TOPK"]),
        "all_inf_targets": int(df.groupby("target_id")["score"].apply(lambda x: np.isinf(x.to_numpy()).all()).sum()),
    }
    qa_rows.append(qa)

    print(f"[{sp}] saved scores: {scores_path}")
    print(f"[{sp}] saved topk  : {topk_path}")
    print(f"[{sp}] QA:", qa)

qa_all = pd.DataFrame(qa_rows)
qa_path = OUT_DIR / "qa_stageC.json"
qa_all.to_json(qa_path, orient="records", indent=2)

print("\n[OK] STAGE C complete.")
print("OUT_DIR:", OUT_DIR)
print("QA JSON:", qa_path)

# IMPORTANT (ID):
# Kalau kamu masih lihat scored_coords=0 dan all_inf_targets=jumlah target,
# itu berarti belum ada coords -> ranking hanya fallback.
# Agar tahap ini benar-benar "accurate scoring", kamu perlu menghasilkan coords di Stage A:
# - tambah DRfold2 repo+models sebagai Kaggle Dataset input, atau
# - tambah external seed dataset (misal Boltz-1 outputs), atau
# - enable gemmi agar TBM bisa ekstrak coords dari CIF.


=== STAGE C INPUTS ===
RUN_DIR : /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac
STD_DIR : /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/standardized/std_3f9797574f
splits  : ['val']
STD_CFG : {'WRITE_FLOAT64': True, 'CLIP_FOR_STD': False, 'CLIP_MIN': -999.999, 'CLIP_MAX': 9999.999}
OUT_DIR: /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/ranking/rank_551d139223
TORCH: True | device: cpu

=== STAGE C: scoring split=val rows=51 ===
[val] scored 51/51 | with_coords_scored=0
[val] saved scores: /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/ranking/rank_551d139223/scores_val.parquet
[val] saved topk  : /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/ranking/rank_551d139223/topk_val.parquet
[val] QA: {'split': 'val', 'rows': 51, 'with_std_coords': 0, 'scored_coords': 0, 'targets': 28, 'topk_rows': 48, 'topk_per_target': 5, 'all_inf_targets': 28}

[OK] STAGE C complete.
OUT_DIR: /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/ranking/rank_551d139223
QA

# GPU Optimization Loop (LBFGS + multi-start + best-checkpoint tracking)

In [4]:
# ============================================================
# NOTEBOOK-2 / STAGE D — GPU Optimization Loop
# (LBFGS + multi-start + best-checkpoint tracking) (ONE CELL)
#
# EN:
# - Load STAGE C topk_{split}.parquet
# - For each candidate with coords: run multi-start LBFGS on a differentiable energy
# - Track best checkpoint per restart and per candidate; save optimized coords
# - Export: opt_manifest_{split}.parquet + qa_stageD.json
#
# ID:
# - Tahap ini melakukan refinement koordinat (post-processing) seperti spirit 1st place:
#   * float64 stabil
#   * LBFGS (autograd) dengan multi-start
#   * best-checkpoint tracking (ambil skor terbaik, bukan iter terakhir)
# - Kalau saat ini belum ada coords, stage ini akan "SKIP gracefully" dan tetap bikin output QA.
# ============================================================

import os, json, time, math, hashlib, warnings
from pathlib import Path
from typing import Dict, Any, Tuple, List, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

# ----------------------------
# 0) Safe helpers
# ----------------------------
def read_json_safe(path: Path) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Missing JSON: {path}")
    txt = path.read_text(encoding="utf-8", errors="replace").strip()
    # avoid "Unexpected token '<' ..."
    if not txt or txt[0] not in "{[":
        raise ValueError(f"Invalid JSON at {path} (starts with {repr(txt[:80])}).")
    return json.loads(txt)

def safe_mkdir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def load_npz_coords(path: Path) -> Optional[np.ndarray]:
    try:
        z = np.load(path, allow_pickle=False)
        if "coords" in z:
            return np.asarray(z["coords"])
        return None
    except Exception:
        return None

def stable_hash_int(s: str) -> int:
    return int(hashlib.sha1(s.encode("utf-8")).hexdigest()[:8], 16)

# ----------------------------
# 1) Locate RUN_DIR / STD_DIR / OUT_DIR (from previous stages)
# ----------------------------
# Use existing globals if present; else autodetect latest
if "STD_DIR" in globals():
    STD_DIR = Path(STD_DIR)
else:
    if "RUN_DIR" in globals():
        RUN_DIR = Path(RUN_DIR)
    else:
        base = Path("/kaggle/working/rna3d_run/candidates")
        if not base.exists():
            raise FileNotFoundError("No /kaggle/working/rna3d_run/candidates. Run Stage A/B/C first.")
        cfg_dirs = sorted([d for d in base.iterdir() if d.is_dir() and d.name.startswith("cfg_")],
                          key=lambda x: x.stat().st_mtime, reverse=True)
        if not cfg_dirs:
            raise FileNotFoundError("No cfg_* folder found. Run Stage A/B/C first.")
        RUN_DIR = cfg_dirs[0]
    std_root = RUN_DIR / "standardized"
    std_dirs = sorted([d for d in std_root.iterdir() if d.is_dir() and d.name.startswith("std_")],
                      key=lambda x: x.stat().st_mtime, reverse=True)
    if not std_dirs:
        raise FileNotFoundError("No std_* folder found. Run Stage B first.")
    STD_DIR = std_dirs[0]

RUN_DIR = STD_DIR.parent.parent
cfg_cand_path = RUN_DIR / "cfg_candidate_gen.json"
CAND_WRAP = read_json_safe(cfg_cand_path)
CFG = CAND_WRAP.get("cfg", {})
splits = list(CFG.get("RUN_SPLITS", ["val"]))

# Stage C OUT_DIR autodetect: latest ranking folder
rank_root = RUN_DIR / "ranking"
rank_dirs = sorted([d for d in rank_root.iterdir() if d.is_dir() and d.name.startswith("rank_")],
                   key=lambda x: x.stat().st_mtime, reverse=True) if rank_root.exists() else []
if not rank_dirs:
    raise FileNotFoundError(f"No rank_* folder found under {rank_root}. Run Stage C first.")
STAGEC_DIR = rank_dirs[0]

print("=== STAGE D INPUTS ===")
print("RUN_DIR   :", RUN_DIR)
print("STD_DIR   :", STD_DIR)
print("STAGEC_DIR:", STAGEC_DIR)
print("splits    :", splits)

# ----------------------------
# 2) Torch / device
# ----------------------------
try:
    import torch
    _HAS_TORCH = True
except Exception:
    _HAS_TORCH = False

if not _HAS_TORCH:
    raise RuntimeError("PyTorch not available. This stage requires torch for LBFGS/autograd.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(True)

print("TORCH:", True, "| device:", device)

# ----------------------------
# 3) Optimization config
# ----------------------------
OPT_CFG = {
    "TOPK_IN": 5,                 # how many candidates per target to optimize (from Stage C topk)
    "MULTI_START": 4,             # includes start-0 = original + (MULTI_START-1) jitter restarts
    "JITTER_STD": 0.50,           # Å
    "LBFGS_MAX_ITER": 120,        # per restart
    "LBFGS_LR": 1.0,
    "LINE_SEARCH": "strong_wolfe",
    "DTYPE": "float64",

    # Energy weights (lower energy is better)
    "BOND_TARGET": 6.0,
    "W_BOND": 1.0,
    "W_SMOOTH": 0.05,
    "W_CLASH": 10.0,
    "CLASH_THRESH": 2.0,
    "MIN_SEQ_SEP": 4,

    # For clash computation:
    "FULL_CDIST_MAX_L": 512,      # if L <= this, compute full cdist; else sampled pairs
    "PAIR_SAMPLES": 30000,        # sampled pairs for big L

    # Centering to remove translation drift:
    "CENTER_EACH_EVAL": True,
}

OPT_ID = hashlib.sha1(json.dumps(OPT_CFG, sort_keys=True).encode()).hexdigest()[:12]
OPT_DIR = safe_mkdir(RUN_DIR / "opt" / f"opt_{OPT_ID}")
(Path(OPT_DIR) / "cfg_stageD_opt.json").write_text(json.dumps({
    "opt_cfg": OPT_CFG,
    "opt_id": OPT_ID,
    "utc_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "std_dir": str(STD_DIR),
    "stagec_dir": str(STAGEC_DIR),
}, indent=2), encoding="utf-8")

print("\n=== OPT CFG ===")
print(json.dumps(OPT_CFG, indent=2))
print("OPT_DIR:", OPT_DIR)

DTYPE = torch.float64 if OPT_CFG["DTYPE"] == "float64" else torch.float32

# ----------------------------
# 4) Energy function (differentiable)
# ----------------------------
def energy_fn(x: torch.Tensor, seed: int) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
    """
    x: (L,3) float64/float32 with grad
    Returns: total_energy, components
    """
    if OPT_CFG["CENTER_EACH_EVAL"]:
        x = x - x.mean(dim=0, keepdim=True)

    L = x.shape[0]

    # bond (consecutive)
    d = torch.linalg.norm(x[1:] - x[:-1], dim=1)
    bond = torch.mean((d - OPT_CFG["BOND_TARGET"]) ** 2)

    # smoothness (second difference)
    if L >= 3:
        dd = x[2:] - 2.0 * x[1:-1] + x[:-2]
        smooth = torch.mean(torch.sum(dd * dd, dim=1))
    else:
        smooth = torch.zeros((), dtype=DTYPE, device=x.device)

    # clash repulsion
    thr = OPT_CFG["CLASH_THRESH"]
    if L <= OPT_CFG["FULL_CDIST_MAX_L"]:
        # full pairwise, mask near neighbors
        dist = torch.cdist(x, x)  # (L,L)
        # mask diagonal and near neighbors
        idx = torch.arange(L, device=x.device)
        sep = (idx[:, None] - idx[None, :]).abs()
        mask = (sep >= OPT_CFG["MIN_SEQ_SEP"])
        # consider only masked entries
        dij = dist[mask]
        clash = torch.mean(torch.clamp(thr - dij, min=0.0) ** 2) if dij.numel() > 0 else torch.zeros((), dtype=DTYPE, device=x.device)
    else:
        # sampled pairs (deterministic)
        g = torch.Generator(device="cpu")
        g.manual_seed(int(seed) & 0x7FFFFFFF)
        M = int(OPT_CFG["PAIR_SAMPLES"])
        i = torch.randint(low=0, high=L - OPT_CFG["MIN_SEQ_SEP"], size=(M,), generator=g, device="cpu")
        off = torch.randint(low=OPT_CFG["MIN_SEQ_SEP"], high=L, size=(M,), generator=g, device="cpu")
        j = (i + off) % L
        i = i.to(x.device)
        j = j.to(x.device)
        dij = torch.linalg.norm(x[i] - x[j], dim=1)
        clash = torch.mean(torch.clamp(thr - dij, min=0.0) ** 2)

    total = (
        OPT_CFG["W_BOND"] * bond
        + OPT_CFG["W_SMOOTH"] * smooth
        + OPT_CFG["W_CLASH"] * clash
    )
    comps = {"bond": bond.detach(), "smooth": smooth.detach(), "clash": clash.detach()}
    return total, comps

# ----------------------------
# 5) LBFGS optimize one start (best-checkpoint tracking)
# ----------------------------
def lbfgs_optimize(coords0: np.ndarray, seed: int) -> Tuple[np.ndarray, float, Dict[str, float], int]:
    """
    Returns best_coords, best_energy, best_components, iters_run
    """
    c0 = np.asarray(coords0)
    L = int(c0.shape[0])
    x = torch.as_tensor(c0, dtype=DTYPE, device=device).clone().detach()
    x.requires_grad_(True)

    opt = torch.optim.LBFGS(
        [x],
        lr=float(OPT_CFG["LBFGS_LR"]),
        max_iter=int(OPT_CFG["LBFGS_MAX_ITER"]),
        line_search_fn=OPT_CFG["LINE_SEARCH"],
    )

    best_e = float("inf")
    best_x = None
    best_comp = {"bond": float("nan"), "smooth": float("nan"), "clash": float("nan")}
    iters_run = 0

    def closure():
        nonlocal best_e, best_x, best_comp, iters_run
        opt.zero_grad(set_to_none=True)
        e, comps = energy_fn(x, seed=seed)
        e.backward()
        iters_run += 1

        ev = float(e.detach().cpu().numpy())
        if ev < best_e:
            best_e = ev
            best_x = x.detach().clone()
            best_comp = {k: float(v.cpu().numpy()) for k, v in comps.items()}
        return e

    try:
        opt.step(closure)
    except Exception as ex:
        # still return best-so-far (or initial)
        pass

    if best_x is None:
        best_x = x.detach().clone()
        e, comps = energy_fn(best_x, seed=seed)
        best_e = float(e.detach().cpu().numpy())
        best_comp = {k: float(v.cpu().numpy()) for k, v in comps.items()}

    out = best_x.detach().cpu().numpy().astype(np.float64, copy=False)
    return out, float(best_e), best_comp, int(iters_run)

# ----------------------------
# 6) Run optimization per split using Stage C topk
# ----------------------------
def save_opt_npz(path: Path, target_id: str, cand_id: str, coords: np.ndarray, meta: Dict[str, Any]):
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez(
        path,
        target_id=str(target_id),
        cand_id=str(cand_id),
        L=np.int32(coords.shape[0]),
        coords=coords.astype(np.float64, copy=False),
        meta=json.dumps(meta),
    )

qa_all = []
for sp in splits:
    topk_path = STAGEC_DIR / f"topk_{sp}.parquet"
    if not topk_path.exists():
        raise FileNotFoundError(f"Missing Stage C topk for split={sp}: {topk_path}")

    df_topk = pd.read_parquet(topk_path)
    df_topk["target_id"] = df_topk["target_id"].astype("string")
    df_topk["cand_id"] = df_topk["cand_id"].astype("string")
    df_topk["std_coords_path"] = df_topk["std_coords_path"].astype("string")
    df_topk["method"] = df_topk["method"].astype("string")

    # ensure max per target
    df_topk = df_topk.sort_values(["target_id", "rank"], ascending=[True, True])
    df_topk = df_topk.groupby("target_id", sort=False).head(int(OPT_CFG["TOPK_IN"])).reset_index(drop=True)

    n_rows = len(df_topk)
    n_with_coords = int((df_topk["std_coords_path"].str.len() > 0).sum())
    print(f"\n=== STAGE D: split={sp} topk_rows={n_rows:,} | with_coords={n_with_coords:,} ===")

    out_rows = []
    skipped_no_coords = 0
    optimized = 0

    for i, r in enumerate(df_topk.itertuples(index=False), 1):
        tid = str(r.target_id)
        cid = str(r.cand_id)
        method = str(r.method)
        stdp = Path(str(r.std_coords_path)) if str(r.std_coords_path) else None

        if stdp is None or (not stdp.exists()):
            skipped_no_coords += 1
            out_rows.append({
                "split": sp, "target_id": tid, "cand_id": cid, "method": method,
                "std_coords_path": str(r.std_coords_path),
                "opt_coords_path": "",
                "ms_id": -1,
                "init_energy": float("inf"),
                "best_energy": float("inf"),
                "iters_run": 0,
                "best_bond": np.nan,
                "best_smooth": np.nan,
                "best_clash": np.nan,
                "status": "SKIP_NO_COORDS",
            })
            continue

        coords0 = load_npz_coords(stdp)
        if coords0 is None:
            skipped_no_coords += 1
            out_rows.append({
                "split": sp, "target_id": tid, "cand_id": cid, "method": method,
                "std_coords_path": str(r.std_coords_path),
                "opt_coords_path": "",
                "ms_id": -1,
                "init_energy": float("inf"),
                "best_energy": float("inf"),
                "iters_run": 0,
                "best_bond": np.nan,
                "best_smooth": np.nan,
                "best_clash": np.nan,
                "status": "SKIP_BAD_NPZ",
            })
            continue

        coords0 = np.asarray(coords0)
        if coords0.ndim != 2 or coords0.shape[1] != 3 or (not np.isfinite(coords0).all()):
            skipped_no_coords += 1
            out_rows.append({
                "split": sp, "target_id": tid, "cand_id": cid, "method": method,
                "std_coords_path": str(r.std_coords_path),
                "opt_coords_path": "",
                "ms_id": -1,
                "init_energy": float("inf"),
                "best_energy": float("inf"),
                "iters_run": 0,
                "best_bond": np.nan,
                "best_smooth": np.nan,
                "best_clash": np.nan,
                "status": "SKIP_INVALID_COORDS",
            })
            continue

        # multi-start
        best_overall_e = float("inf")
        best_overall_path = ""
        best_overall_comp = {"bond": np.nan, "smooth": np.nan, "clash": np.nan}
        best_overall_ms = -1
        best_overall_iters = 0

        # init energy (for logging only)
        seed0 = stable_hash_int(f"{tid}|{cid}|{sp}|init")
        with torch.no_grad():
            x0 = torch.as_tensor(coords0, dtype=DTYPE, device=device)
            e0, _ = energy_fn(x0, seed=seed0)
            init_e = float(e0.detach().cpu().numpy())

        for ms in range(int(OPT_CFG["MULTI_START"])):
            seed = stable_hash_int(f"{tid}|{cid}|{sp}|ms{ms}")
            if ms == 0:
                start = coords0
            else:
                rng = np.random.default_rng(seed)
                start = coords0 + rng.normal(0.0, float(OPT_CFG["JITTER_STD"]), size=coords0.shape).astype(np.float64)

            opt_coords, best_e, best_comp, iters_run = lbfgs_optimize(start, seed=seed)

            # save each restart result (optional but helpful)
            out_dir = safe_mkdir(OPT_DIR / sp / tid)
            out_npz = out_dir / f"{cid}_ms{ms:02d}.npz"
            save_opt_npz(out_npz, tid, cid, opt_coords, meta={
                "split": sp, "method": method, "ms_id": ms,
                "init_energy": init_e, "best_energy": best_e,
                "best_comp": best_comp, "iters_run": iters_run,
                "std_coords_path": str(stdp),
                "opt_cfg": OPT_CFG,
            })

            out_rows.append({
                "split": sp, "target_id": tid, "cand_id": cid, "method": method,
                "std_coords_path": str(stdp),
                "opt_coords_path": str(out_npz),
                "ms_id": int(ms),
                "init_energy": float(init_e),
                "best_energy": float(best_e),
                "iters_run": int(iters_run),
                "best_bond": float(best_comp.get("bond", np.nan)),
                "best_smooth": float(best_comp.get("smooth", np.nan)),
                "best_clash": float(best_comp.get("clash", np.nan)),
                "status": "OK",
            })

            if best_e < best_overall_e:
                best_overall_e = best_e
                best_overall_path = str(out_npz)
                best_overall_comp = best_comp
                best_overall_ms = ms
                best_overall_iters = iters_run

        # record best pointer (one row per candidate)
        out_rows.append({
            "split": sp, "target_id": tid, "cand_id": cid, "method": method,
            "std_coords_path": str(stdp),
            "opt_coords_path": str(best_overall_path),
            "ms_id": int(best_overall_ms),
            "init_energy": float(init_e),
            "best_energy": float(best_overall_e),
            "iters_run": int(best_overall_iters),
            "best_bond": float(best_overall_comp.get("bond", np.nan)),
            "best_smooth": float(best_overall_comp.get("smooth", np.nan)),
            "best_clash": float(best_overall_comp.get("clash", np.nan)),
            "status": "BEST",
        })
        optimized += 1

        if i % 10 == 0 or i == n_rows:
            print(f"[{sp}] processed {i:,}/{n_rows:,} | optimized={optimized:,} | skipped={skipped_no_coords:,}", end="\r")

    print()

    df_opt = pd.DataFrame(out_rows)
    opt_manifest_path = OPT_DIR / f"opt_manifest_{sp}.parquet"
    df_opt.to_parquet(opt_manifest_path, index=False)

    qa = {
        "split": sp,
        "topk_rows_in": int(n_rows),
        "with_coords_in": int(n_with_coords),
        "optimized_candidates": int(optimized),
        "skipped_no_coords": int(skipped_no_coords),
        "opt_rows_written": int(len(df_opt)),
        "best_rows": int((df_opt["status"] == "BEST").sum()),
        "device": str(device),
    }
    qa_all.append(qa)

    print(f"[{sp}] saved opt manifest: {opt_manifest_path}")
    print(f"[{sp}] QA:", qa)

qa_path = OPT_DIR / "qa_stageD.json"
Path(qa_path).write_text(json.dumps(qa_all, indent=2), encoding="utf-8")

print("\n[OK] STAGE D complete.")
print("OPT_DIR:", OPT_DIR)
print("QA    :", qa_path)

# NOTE (ID):
# Kalau QA menunjukkan optimized_candidates=0 dan skipped_no_coords tinggi,
# itu normal karena kandidat kamu belum punya koordinat.
# Begitu kamu punya coords dari DRfold2 / external seeds / gemmi TBM,
# stage ini otomatis mulai mengoptimasi dan menyimpan NPZ hasilnya.


=== STAGE D INPUTS ===
RUN_DIR   : /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac
STD_DIR   : /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/standardized/std_3f9797574f
STAGEC_DIR: /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/ranking/rank_551d139223
splits    : ['val']
TORCH: True | device: cpu

=== OPT CFG ===
{
  "TOPK_IN": 5,
  "MULTI_START": 4,
  "JITTER_STD": 0.5,
  "LBFGS_MAX_ITER": 120,
  "LBFGS_LR": 1.0,
  "LINE_SEARCH": "strong_wolfe",
  "DTYPE": "float64",
  "BOND_TARGET": 6.0,
  "W_BOND": 1.0,
  "W_SMOOTH": 0.05,
  "W_CLASH": 10.0,
  "CLASH_THRESH": 2.0,
  "MIN_SEQ_SEP": 4,
  "FULL_CDIST_MAX_L": 512,
  "PAIR_SAMPLES": 30000,
  "CENTER_EACH_EVAL": true
}
OPT_DIR: /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/opt/opt_1b6cd751e419

=== STAGE D: split=val topk_rows=48 | with_coords=0 ===

[val] saved opt manifest: /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/opt/opt_1b6cd751e419/opt_manifest_val.parquet
[val] QA: {'split': 'val', 'topk_row

# Final Training 

In [5]:
# ============================================================
# NOTEBOOK-2 / STAGE E — FINAL TRAINING (Ranker) + Bundle Export (ONE CELL)
#
# Goal (EN):
# - Train a robust candidate ranker (Learning-to-Rank) using float64, stable scoring targets,
#   and geometry features (torch.cdist when possible).
# - Uses multi-reference labels on VAL (n_ref=40) and single-reference labels on TRAIN (n_ref=1) if available.
# - Exports a portable bundle for the Submission notebook:
#   ranker_model + feature_cols + cfg + reports
#
# Penjelasan (ID):
# - Ini adalah "training" ranker untuk memilih kandidat terbaik Top-5 per target secara konsisten.
# - Sesuai spirit 1st place: float64, ranking stabil, dan fitur geometri yang kuat.
# - Aman walaupun kandidat coords belum ada (akan tetap membuat file output & report, tapi tidak bisa melatih).
# ============================================================

import os, json, time, math, hashlib, warnings
from pathlib import Path
from typing import Dict, Any, Optional, Tuple, List

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

# ----------------------------
# 0) Safe helpers (avoid invalid JSON / HTML)
# ----------------------------
def read_json_safe(path: Path) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Missing JSON: {path}")
    txt = path.read_text(encoding="utf-8", errors="replace").strip()
    if not txt or txt[0] not in "{[":
        raise ValueError(f"Invalid JSON at {path} (starts with {repr(txt[:80])})")
    return json.loads(txt)

def safe_mkdir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def stable_hash_int(s: str) -> int:
    return int(hashlib.sha1(s.encode("utf-8")).hexdigest()[:8], 16)

def pick_latest_dir(parent: Path, prefix: str) -> Path:
    if not parent.exists():
        return Path("")
    ds = [d for d in parent.iterdir() if d.is_dir() and d.name.startswith(prefix)]
    if not ds:
        return Path("")
    ds.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    return ds[0]

def load_npz_any_coords(npz_path: Path) -> Optional[np.ndarray]:
    try:
        z = np.load(npz_path, allow_pickle=False)
        # prefer 'coords'
        if "coords" in z:
            c = np.asarray(z["coords"])
            if c.ndim == 2 and c.shape[1] == 3:
                return c
            if c.ndim == 3 and c.shape[-1] == 3:
                # (n_ref,L,3) -> not candidate format, ignore here
                return None
        # fallback: search any array shaped (L,3)
        for k in z.files:
            arr = np.asarray(z[k])
            if arr.ndim == 2 and arr.shape[1] == 3 and np.isfinite(arr).all():
                return arr
        return None
    except Exception:
        return None

def load_label_refs(npz_path: Path) -> Optional[np.ndarray]:
    """
    Returns labels as (R, L, 3) float64 (multi-ref).
    """
    try:
        z = np.load(npz_path, allow_pickle=False)
        # common key in our pipeline: 'coords'
        cand = None
        if "coords" in z:
            cand = np.asarray(z["coords"])
        else:
            # fallback: find any (R,L,3)
            for k in z.files:
                arr = np.asarray(z[k])
                if arr.ndim == 3 and arr.shape[-1] == 3:
                    cand = arr
                    break
        if cand is None or cand.ndim != 3 or cand.shape[-1] != 3:
            return None
        cand = cand.astype(np.float64, copy=False)
        if not np.isfinite(cand).all():
            return None
        return cand
    except Exception:
        return None

# ----------------------------
# 1) Locate RUN_DIR / ART_ROOT / STD_DIR / STAGEC_DIR / OPT_DIR
# ----------------------------
if "RUN_DIR" in globals():
    RUN_DIR = Path(RUN_DIR)
else:
    base = Path("/kaggle/working/rna3d_run/candidates")
    if not base.exists():
        raise FileNotFoundError("No /kaggle/working/rna3d_run/candidates. Run Stages A–D first.")
    cfg_dirs = sorted([d for d in base.iterdir() if d.is_dir() and d.name.startswith("cfg_")],
                      key=lambda x: x.stat().st_mtime, reverse=True)
    if not cfg_dirs:
        raise FileNotFoundError("No cfg_* folder found. Run Stages A–D first.")
    RUN_DIR = cfg_dirs[0]

cfg_cand_path = RUN_DIR / "cfg_candidate_gen.json"
CAND_WRAP = read_json_safe(cfg_cand_path)
CFG_A = CAND_WRAP.get("cfg", {})
splits = list(CFG_A.get("RUN_SPLITS", ["val"]))

ART_ROOT = Path(CAND_WRAP.get("art_root", ""))
if not ART_ROOT.exists():
    raise FileNotFoundError(f"ART_ROOT missing: {ART_ROOT}")

STD_ROOT = RUN_DIR / "standardized"
STD_DIR = pick_latest_dir(STD_ROOT, "std_")
if not STD_DIR:
    raise FileNotFoundError(f"No std_* found under {STD_ROOT}. Run Stage B first.")

STAGEC_ROOT = RUN_DIR / "ranking"
STAGEC_DIR = pick_latest_dir(STAGEC_ROOT, "rank_")
if not STAGEC_DIR:
    raise FileNotFoundError(f"No rank_* found under {STAGEC_ROOT}. Run Stage C first.")

OPT_ROOT = RUN_DIR / "opt"
OPT_DIR = pick_latest_dir(OPT_ROOT, "opt_")  # may be empty if Stage D not run
# OPT_DIR can be "" (no stage D) -> handled.

print("=== STAGE E INPUTS ===")
print("RUN_DIR   :", RUN_DIR)
print("ART_ROOT  :", ART_ROOT)
print("STD_DIR   :", STD_DIR)
print("STAGEC_DIR:", STAGEC_DIR)
print("OPT_DIR   :", OPT_DIR if str(OPT_DIR) else "(none)")
print("splits    :", splits)

# Labels directories
LBL_TRAIN_DIR = ART_ROOT / "labels_npz" / "train"
LBL_VAL_DIR   = ART_ROOT / "labels_npz" / "val"

# ----------------------------
# 2) Torch setup (for fast distance features)
# ----------------------------
try:
    import torch
    _HAS_TORCH = True
except Exception:
    _HAS_TORCH = False

device = None
if _HAS_TORCH:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.set_grad_enabled(False)

print("TORCH:", _HAS_TORCH, "| device:", str(device) if device is not None else "none")

# ----------------------------
# 3) Kabsch RMSD (float64) for supervision (multi-reference)
# ----------------------------
def kabsch_rmsd(P: np.ndarray, Q: np.ndarray) -> float:
    """
    P, Q: (L,3) float64
    Returns RMSD after optimal rigid alignment.
    """
    P = P.astype(np.float64, copy=False)
    Q = Q.astype(np.float64, copy=False)
    if P.shape != Q.shape or P.ndim != 2 or P.shape[1] != 3:
        return float("inf")
    if not (np.isfinite(P).all() and np.isfinite(Q).all()):
        return float("inf")

    Pc = P - P.mean(axis=0, keepdims=True)
    Qc = Q - Q.mean(axis=0, keepdims=True)
    C = Pc.T @ Qc  # (3,3)
    try:
        V, S, Wt = np.linalg.svd(C)
        d = np.sign(np.linalg.det(V @ Wt))
        D = np.diag([1.0, 1.0, d])
        U = V @ D @ Wt
        P_rot = Pc @ U
        diff = P_rot - Qc
        return float(np.sqrt(np.mean(np.sum(diff * diff, axis=1)) + 1e-12))
    except Exception:
        return float("inf")

def best_ref_rmsd(candidate: np.ndarray, refs: np.ndarray) -> float:
    """
    candidate: (L,3)
    refs: (R,L,3)
    Returns min RMSD over references.
    """
    if refs is None or refs.ndim != 3:
        return float("inf")
    best = float("inf")
    for r in range(refs.shape[0]):
        v = kabsch_rmsd(candidate, refs[r])
        if v < best:
            best = v
    return best

# ----------------------------
# 4) Geometry feature extraction (float64 + torch.cdist where possible)
# ----------------------------
FEAT_CFG = {
    "PAIR_SAMPLES": 20000,   # for long sequences
    "FULL_CDIST_MAX_L": 512,
    "MIN_SEQ_SEP": 4,
    "CLASH_THRESH": 2.0,
}
def geom_features(coords: np.ndarray, seed: int) -> Dict[str, float]:
    """
    Strong cheap features:
    - bond mean/std/p01/p99
    - curvature mean
    - radius of gyration
    - clash rate (dist < thresh for long-range pairs)
    Uses torch.cdist for L<=FULL_CDIST_MAX_L (if torch available), else deterministic sampling.
    """
    c = np.asarray(coords)
    L = int(c.shape[0])
    out = {
        "L": float(L),
        "bond_mean": np.nan, "bond_std": np.nan, "bond_p01": np.nan, "bond_p99": np.nan,
        "curv_mean": np.nan,
        "rg": np.nan,
        "clash_rate": np.nan,
        "coord_absmax": np.nan,
    }
    if c.ndim != 2 or c.shape[1] != 3 or L < 2 or (not np.isfinite(c).all()):
        return out

    c64 = c.astype(np.float64, copy=False)
    out["coord_absmax"] = float(np.max(np.abs(c64)))

    # bond stats
    d = np.linalg.norm(c64[1:] - c64[:-1], axis=1)
    out["bond_mean"] = float(np.mean(d))
    out["bond_std"]  = float(np.std(d))
    out["bond_p01"]  = float(np.quantile(d, 0.01))
    out["bond_p99"]  = float(np.quantile(d, 0.99))

    # curvature
    if L >= 3:
        dd = c64[2:] - 2.0*c64[1:-1] + c64[:-2]
        out["curv_mean"] = float(np.mean(np.sum(dd*dd, axis=1)))
    else:
        out["curv_mean"] = 0.0

    # radius of gyration
    cen = np.mean(c64, axis=0, keepdims=True)
    out["rg"] = float(np.sqrt(np.mean(np.sum((c64 - cen)**2, axis=1)) + 1e-12))

    # clash rate (long-range)
    thr = float(FEAT_CFG["CLASH_THRESH"])
    min_sep = int(FEAT_CFG["MIN_SEQ_SEP"])

    if _HAS_TORCH and L <= int(FEAT_CFG["FULL_CDIST_MAX_L"]):
        tc = torch.as_tensor(c64, dtype=torch.float64, device=device)
        dist = torch.cdist(tc, tc)  # (L,L)
        idx = torch.arange(L, device=device)
        sep = (idx[:, None] - idx[None, :]).abs()
        mask = (sep >= min_sep)
        dij = dist[mask]
        if dij.numel() > 0:
            out["clash_rate"] = float((dij < thr).double().mean().detach().cpu().numpy())
        else:
            out["clash_rate"] = 0.0
    else:
        rng = np.random.default_rng(seed)
        M = int(FEAT_CFG["PAIR_SAMPLES"])
        if L <= min_sep + 1:
            out["clash_rate"] = 0.0
        else:
            i = rng.integers(0, L - min_sep, size=M)
            off = rng.integers(min_sep, L, size=M)
            j = (i + off) % L
            dij = np.linalg.norm(c64[i] - c64[j], axis=1)
            out["clash_rate"] = float(np.mean(dij < thr))

    return out

# ----------------------------
# 5) Load candidates (prefer optimized BEST if available; else std coords)
# ----------------------------
def load_topk_base(split: str) -> pd.DataFrame:
    topk_p = STAGEC_DIR / f"topk_{split}.parquet"
    if not topk_p.exists():
        raise FileNotFoundError(f"Missing Stage C topk: {topk_p}")
    df = pd.read_parquet(topk_p)
    # expected columns: target_id, cand_id, method, rank, std_coords_path, (maybe score cols)
    for col in ["target_id","cand_id","method"]:
        if col in df.columns:
            df[col] = df[col].astype("string")
    if "rank" in df.columns:
        df["rank"] = pd.to_numeric(df["rank"], errors="coerce").fillna(999).astype("int32")
    if "std_coords_path" in df.columns:
        df["std_coords_path"] = df["std_coords_path"].astype("string")
    else:
        df["std_coords_path"] = ""
    return df

def load_opt_best(split: str) -> Optional[pd.DataFrame]:
    if not str(OPT_DIR):
        return None
    p = OPT_DIR / f"opt_manifest_{split}.parquet"
    if not p.exists():
        return None
    df = pd.read_parquet(p)
    if "status" not in df.columns:
        return None
    df = df[df["status"].astype("string") == "BEST"].copy()
    if len(df) == 0:
        return None
    for col in ["target_id","cand_id","method","opt_coords_path","std_coords_path"]:
        if col in df.columns:
            df[col] = df[col].astype("string")
    return df

# ----------------------------
# 6) Build training table with supervision (RMSD to labels) + features
# ----------------------------
TRAIN_CFG = {
    "TOPK_PER_TARGET": 5,         # from Stage C
    "MAX_L_FOR_SUP": 6000,        # keep supervision compute safe; val max is ~4640
    "USE_TRAIN_SPLIT_IF_PRESENT": False,  # set True only if you have candidates+coords for train
}
TRAIN_ID = hashlib.sha1(json.dumps(TRAIN_CFG, sort_keys=True).encode()).hexdigest()[:10]

BUNDLE_DIR = safe_mkdir(Path("/kaggle/working/rna3d_final_bundle") / f"bundle_{TRAIN_ID}")
safe_mkdir(BUNDLE_DIR / "reports")
safe_mkdir(BUNDLE_DIR / "models")

(Path(BUNDLE_DIR) / "train_cfg.json").write_text(json.dumps({
    "train_cfg": TRAIN_CFG,
    "feat_cfg": FEAT_CFG,
    "utc_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "run_dir": str(RUN_DIR),
    "stagec_dir": str(STAGEC_DIR),
    "opt_dir": str(OPT_DIR) if str(OPT_DIR) else "",
}, indent=2), encoding="utf-8")

def build_split_table(split: str) -> pd.DataFrame:
    base = load_topk_base(split)

    # merge optimized best if available
    opt_best = load_opt_best(split)
    if opt_best is not None:
        opt_best = opt_best[["target_id","cand_id","opt_coords_path","best_energy","best_bond","best_smooth","best_clash"]].copy() \
            if set(["best_energy","best_bond","best_smooth","best_clash"]).issubset(opt_best.columns) \
            else opt_best[["target_id","cand_id","opt_coords_path"]].copy()
        opt_best = opt_best.rename(columns={"opt_coords_path":"best_coords_path"})
        base = base.merge(opt_best, on=["target_id","cand_id"], how="left")
    else:
        base["best_coords_path"] = ""

    # choose final coords path for feature/supervision
    base["coords_path_used"] = base.get("best_coords_path","").astype("string")
    m = base["coords_path_used"].str.len() == 0
    base.loc[m, "coords_path_used"] = base["std_coords_path"].astype("string")

    # load labels refs for supervision (train/val only)
    if split == "train":
        lbl_dir = LBL_TRAIN_DIR
    elif split == "val":
        lbl_dir = LBL_VAL_DIR
    else:
        lbl_dir = None

    rows = []
    for r in base.itertuples(index=False):
        tid = str(getattr(r, "target_id"))
        cid = str(getattr(r, "cand_id"))
        method = str(getattr(r, "method"))
        rank_in = int(getattr(r, "rank", 999))
        coords_path = str(getattr(r, "coords_path_used", ""))

        # candidate coords
        coords = None
        if coords_path and Path(coords_path).exists():
            coords = load_npz_any_coords(Path(coords_path))

        has_coords = (coords is not None)

        # geometry features
        if has_coords:
            seed = stable_hash_int(f"{split}|{tid}|{cid}")
            gf = geom_features(coords, seed=seed)
        else:
            gf = geom_features(np.zeros((2,3), dtype=np.float64), seed=0)
            for k in gf:
                if k != "L":
                    gf[k] = np.nan
            gf["L"] = float(getattr(r, "L_expected", np.nan)) if hasattr(r, "L_expected") else np.nan

        # supervision: best RMSD over refs
        y_rmsd = np.nan
        if lbl_dir is not None and has_coords:
            if coords.shape[0] <= int(TRAIN_CFG["MAX_L_FOR_SUP"]):
                lbl_path = lbl_dir / f"{tid}.npz"
                if lbl_path.exists():
                    refs = load_label_refs(lbl_path)
                    if refs is not None and refs.shape[1] == coords.shape[0]:
                        y_rmsd = best_ref_rmsd(coords, refs)

        # gather stageC score fields if present
        score = float(getattr(r, "score", np.nan)) if hasattr(r, "score") else np.nan
        sbond = float(getattr(r, "score_bond", np.nan)) if hasattr(r, "score_bond") else np.nan
        scurv = float(getattr(r, "score_curv", np.nan)) if hasattr(r, "score_curv") else np.nan
        sclash = float(getattr(r, "score_clash", np.nan)) if hasattr(r, "score_clash") else np.nan
        srg = float(getattr(r, "score_rg", np.nan)) if hasattr(r, "score_rg") else np.nan

        out = {
            "split": split,
            "target_id": tid,
            "cand_id": cid,
            "method": method,
            "rank_in": rank_in,
            "coords_path_used": coords_path,
            "has_coords": bool(has_coords),
            "y_rmsd_bestref": float(y_rmsd) if np.isfinite(y_rmsd) else np.nan,
            "stageC_score": score,
            "stageC_bond": sbond,
            "stageC_curv": scurv,
            "stageC_clash": sclash,
            "stageC_rg": srg,
        }
        out.update(gf)

        # include opt energy if exists
        for optk in ["best_energy","best_bond","best_smooth","best_clash"]:
            if hasattr(r, optk):
                out[f"opt_{optk}"] = float(getattr(r, optk))
            else:
                out[f"opt_{optk}"] = np.nan

        rows.append(out)

    df = pd.DataFrame(rows)
    return df

# Build for available splits
dfs = []
available_splits = []
for sp in splits:
    if sp not in ["train","val","test"]:
        continue
    # training uses train/val; test has no labels (still can build features later)
    df_sp = build_split_table(sp)
    dfs.append(df_sp)
    available_splits.append(sp)

df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
table_path = BUNDLE_DIR / "reports" / "rank_train_table.parquet"
df_all.to_parquet(table_path, index=False)

print("\n=== TRAIN TABLE BUILT ===")
print("available_splits:", available_splits)
print("rows:", len(df_all))
print("has_coords:", int(df_all["has_coords"].sum()) if len(df_all) else 0)
print("has_supervision(y_rmsd notna):", int(df_all["y_rmsd_bestref"].notna().sum()) if len(df_all) else 0)
print("saved:", table_path)

# If no supervision, stop safely (still exported table)
if len(df_all) == 0 or int(df_all["y_rmsd_bestref"].notna().sum()) == 0:
    msg = (
        "\n[STOP - SAFE]\n"
        "No supervised labels could be computed (likely because candidates have no coords yet).\n"
        "Add coordinate-generating sources (DRfold2 / external seeds / gemmi TBM coords), then rerun Stages A–E.\n"
    )
    (BUNDLE_DIR / "reports" / "train_stop_reason.txt").write_text(msg, encoding="utf-8")
    print(msg)
else:
    # ----------------------------
    # 7) Prepare LTR labels (relevance) + features
    # ----------------------------
    # We train on split='val' by default (robust, small). If you later generate train candidates, switch.
    train_df = df_all[df_all["split"] == "val"].copy() if "val" in df_all["split"].unique() else df_all.copy()

    # Keep only rows with supervision
    train_df = train_df[train_df["y_rmsd_bestref"].notna()].copy()

    # Build relevance per target: best RMSD -> highest relevance
    train_df["y_rank"] = train_df.groupby("target_id")["y_rmsd_bestref"].rank(method="first", ascending=True).astype("int32")
    # relevance: top1=4, top2=3, top3=2, top4=1, else 0 (listwise training)
    train_df["relevance"] = (5 - train_df["y_rank"]).clip(lower=0, upper=4).astype("int32")

    # Feature columns
    # numeric features (float64)
    feature_cols = [
        "rank_in",
        "stageC_score","stageC_bond","stageC_curv","stageC_clash","stageC_rg",
        "L","bond_mean","bond_std","bond_p01","bond_p99","curv_mean","rg","clash_rate","coord_absmax",
        "opt_best_energy","opt_best_bond","opt_best_smooth","opt_best_clash",
    ]
    # ensure existing
    feature_cols = [c for c in feature_cols if c in train_df.columns]

    # One-hot method
    methods = sorted(train_df["method"].astype("string").fillna("UNK").unique().tolist())
    for m in methods:
        col = f"method__{m}"
        train_df[col] = (train_df["method"] == m).astype("int32")
        feature_cols.append(col)

    # Fill NaNs
    X = train_df[feature_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    for c in X.columns:
        if X[c].dtype.kind in "biu":
            X[c] = X[c].fillna(0)
        else:
            X[c] = X[c].astype(np.float64, copy=False).fillna(X[c].median())

    y = train_df["relevance"].astype("int32").to_numpy()
    group_sizes = train_df.groupby("target_id").size().to_numpy()

    # ----------------------------
    # 8) Train Ranker (LightGBM preferred; fallback to XGBoost if missing)
    # ----------------------------
    model = None
    report = {"backend": None}

    # LightGBM ranker
    try:
        import lightgbm as lgb
        report["backend"] = "lightgbm"
        params = dict(
            objective="lambdarank",
            metric="ndcg",
            ndcg_eval_at=[1,3,5],
            learning_rate=0.05,
            num_leaves=63,
            min_data_in_leaf=10,
            feature_fraction=0.9,
            bagging_fraction=0.9,
            bagging_freq=1,
            lambda_l2=1.0,
            verbosity=-1,
            seed=42,
        )
        lgb_train = lgb.Dataset(X, label=y, group=group_sizes)
        model = lgb.train(params, lgb_train, num_boost_round=1200)
        # quick self-metric (train-only, since val is tiny)
        pred = model.predict(X)
        train_df["pred_score"] = pred
        # ndcg@5 per target (manual)
        def ndcg_at_k(rels, k=5):
            rels = np.asarray(rels, dtype=np.float64)
            rels = rels[:k]
            dcg = np.sum((2**rels - 1) / np.log2(np.arange(2, 2+len(rels))))
            ideal = np.sort(rels)[::-1]
            idcg = np.sum((2**ideal - 1) / np.log2(np.arange(2, 2+len(ideal))))
            return float(dcg / (idcg + 1e-12))
        ndcgs = []
        for tid, sub in train_df.groupby("target_id", sort=False):
            sub = sub.sort_values("pred_score", ascending=False)
            ndcgs.append(ndcg_at_k(sub["relevance"].to_numpy(), k=5))
        report["train_ndcg5_mean"] = float(np.mean(ndcgs))
        report["train_ndcg5_min"] = float(np.min(ndcgs))
        report["train_ndcg5_max"] = float(np.max(ndcgs))

        model_path = BUNDLE_DIR / "models" / "ranker_model_lgb.txt"
        model.save_model(str(model_path))
        report["model_path"] = str(model_path)

    except Exception as e_lgb:
        # XGBoost fallback
        try:
            import xgboost as xgb
            report["backend"] = "xgboost"
            # XGBoost expects group info via set_group
            dtrain = xgb.DMatrix(X, label=y)
            dtrain.set_group(group_sizes.astype(np.uint32))
            params = dict(
                objective="rank:pairwise",
                eval_metric="ndcg@5",
                eta=0.05,
                max_depth=6,
                min_child_weight=5,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_lambda=1.0,
                seed=42,
            )
            model = xgb.train(params, dtrain, num_boost_round=1200)
            pred = model.predict(dtrain)
            train_df["pred_score"] = pred
            # ndcg@5 (manual)
            def ndcg_at_k(rels, k=5):
                rels = np.asarray(rels, dtype=np.float64)
                rels = rels[:k]
                dcg = np.sum((2**rels - 1) / np.log2(np.arange(2, 2+len(rels))))
                ideal = np.sort(rels)[::-1]
                idcg = np.sum((2**ideal - 1) / np.log2(np.arange(2, 2+len(ideal))))
                return float(dcg / (idcg + 1e-12))
            ndcgs = []
            for tid, sub in train_df.groupby("target_id", sort=False):
                sub = sub.sort_values("pred_score", ascending=False)
                ndcgs.append(ndcg_at_k(sub["relevance"].to_numpy(), k=5))
            report["train_ndcg5_mean"] = float(np.mean(ndcgs))
            report["train_ndcg5_min"] = float(np.min(ndcgs))
            report["train_ndcg5_max"] = float(np.max(ndcgs))

            model_path = BUNDLE_DIR / "models" / "ranker_model_xgb.json"
            model.save_model(str(model_path))
            report["model_path"] = str(model_path)
        except Exception as e_xgb:
            report["backend"] = "none"
            report["error_lgb"] = str(e_lgb)
            report["error_xgb"] = str(e_xgb)
            raise RuntimeError(
                "Could not train ranker: LightGBM and XGBoost both unavailable."
            )

    # Save feature cols + report
    (BUNDLE_DIR / "feature_cols.json").write_text(json.dumps(feature_cols, indent=2), encoding="utf-8")
    (BUNDLE_DIR / "method_categories.json").write_text(json.dumps(methods, indent=2), encoding="utf-8")
    (BUNDLE_DIR / "reports" / "train_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

    print("\n=== TRAIN DONE ===")
    print("backend:", report["backend"])
    print("train_ndcg5_mean:", report.get("train_ndcg5_mean", None))
    print("bundle:", BUNDLE_DIR)

    # ----------------------------
    # 9) Produce "final_top5" for splits that exist (using ranker preds)
    # ----------------------------
    def apply_ranker_to_split(split: str) -> pd.DataFrame:
        df = df_all[df_all["split"] == split].copy()
        if len(df) == 0:
            return df

        # Rebuild same feature columns (including method OHE)
        methods_local = sorted(df["method"].astype("string").fillna("UNK").unique().tolist())
        # make sure all global methods columns exist
        for m in methods:
            col = f"method__{m}"
            df[col] = (df["method"] == m).astype("int32")

        Xs = df[[c for c in feature_cols if c in df.columns]].copy()
        Xs = Xs.replace([np.inf, -np.inf], np.nan)
        for c in Xs.columns:
            if Xs[c].dtype.kind in "biu":
                Xs[c] = Xs[c].fillna(0)
            else:
                Xs[c] = Xs[c].astype(np.float64, copy=False).fillna(Xs[c].median())

        # predict
        if report["backend"] == "lightgbm":
            pred = model.predict(Xs)
        else:
            import xgboost as xgb
            pred = model.predict(xgb.DMatrix(Xs))
        df["ranker_score"] = pred

        # final rank per target: higher ranker_score better
        df = df.sort_values(["target_id","ranker_score","rank_in","cand_id"], ascending=[True, False, True, True])
        df["final_rank"] = df.groupby("target_id").cumcount() + 1
        df_top5 = df[df["final_rank"] <= 5].copy()
        return df_top5

    for sp in available_splits:
        df_top5 = apply_ranker_to_split(sp)
        outp = BUNDLE_DIR / "reports" / f"final_top5_{sp}.parquet"
        df_top5.to_parquet(outp, index=False)
        print(f"[OK] saved final_top5_{sp}: {outp}")

print("\n[OK] STAGE E complete.")
print("BUNDLE_DIR:", BUNDLE_DIR)
print("Next (submission notebook): load bundle + run candidate gen (test) + standardize + ranker select top5 + (optional) optimize + export submission.")


=== STAGE E INPUTS ===
RUN_DIR   : /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac
ART_ROOT  : /kaggle/input/stanford-rna-3d-folding-part-2-dataset/rna3d_artifacts_v1
STD_DIR   : /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/standardized/std_3f9797574f
STAGEC_DIR: /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/ranking/rank_551d139223
OPT_DIR   : /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/opt/opt_1b6cd751e419
splits    : ['val']
TORCH: True | device: cpu

=== TRAIN TABLE BUILT ===
available_splits: ['val']
rows: 48
has_coords: 0
has_supervision(y_rmsd notna): 0
saved: /kaggle/working/rna3d_final_bundle/bundle_0462b2dfe9/reports/rank_train_table.parquet

[STOP - SAFE]
No supervised labels could be computed (likely because candidates have no coords yet).
Add coordinate-generating sources (DRfold2 / external seeds / gemmi TBM coords), then rerun Stages A–E.


[OK] STAGE E complete.
BUNDLE_DIR: /kaggle/working/rna3d_final_bundle/bundle_0462b2dfe9
Next (submis

# Final Selection (Top-5) + Save Best Model Bundle 

In [6]:
# ============================================================
# NOTEBOOK-2 / STAGE F — Final Selection (Top-5) + Save Best Model Bundle (REVISI FULL v2) (ONE CELL)
#
# FIX v2:
# - Prevent TypeError: "arg must be a list, tuple, 1-d array, or Series"
#   Root cause is almost always DUPLICATE COLUMN NAMES -> df[col] becomes DataFrame (2D), not Series.
# - This revision:
#   * Deduplicates df.columns safely (keep first)
#   * Deduplicates feature_cols (preserve order)
#   * Builds X via reindex(columns=feature_cols) to guarantee 1D Series per col
#   * Robust numeric fill: handles object/strings safely
#
# Output:
# - RUN_DIR/final_selection/final_<id>/final_top5_{split}.parquet + .csv
# - RUN_DIR/model_bundle_best + model_bundle_best.zip
# ============================================================

import os, json, time, shutil, hashlib, warnings, zipfile
from pathlib import Path
from typing import Dict, Any, Optional, Tuple, List

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

# ----------------------------
# 0) helpers
# ----------------------------
def safe_mkdir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def read_json_safe(path: Path) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Missing JSON: {path}")
    txt = path.read_text(encoding="utf-8", errors="replace").strip()
    # avoid "Unexpected token '<' ... not valid JSON"
    if not txt or txt[0] not in "{[":
        raise ValueError(f"Invalid JSON at {path} (starts with {repr(txt[:80])})")
    return json.loads(txt)

def stable_hash_int(s: str) -> int:
    return int(hashlib.sha1(s.encode("utf-8")).hexdigest()[:8], 16)

def pick_latest_dir(parent: Path, prefix: str) -> Path:
    if not parent.exists():
        return Path("")
    ds = [d for d in parent.iterdir() if d.is_dir() and d.name.startswith(prefix)]
    if not ds:
        return Path("")
    ds.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    return ds[0]

def load_npz_coords(npz_path: Path) -> Optional[np.ndarray]:
    try:
        z = np.load(npz_path, allow_pickle=False)
        if "coords" in z:
            c = np.asarray(z["coords"])
            if c.ndim == 2 and c.shape[1] == 3 and np.isfinite(c).all():
                return c
        for k in z.files:
            arr = np.asarray(z[k])
            if arr.ndim == 2 and arr.shape[1] == 3 and np.isfinite(arr).all():
                return arr
        return None
    except Exception:
        return None

def copy_if_exists(src: Path, dst: Path):
    try:
        if src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
    except Exception:
        pass

def unique_preserve(seq: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

# fallback priority if no ranker
def method_priority(method: str) -> int:
    m = str(method).upper()
    if m == "EXT": return 0
    if m == "DRFOLD2": return 1
    if m == "TBM": return 2
    if m == "TBM_REF": return 3
    return 9

# ----------------------------
# 1) locate RUN_DIR / stage dirs
# ----------------------------
if "RUN_DIR" in globals():
    RUN_DIR = Path(RUN_DIR)
else:
    base = Path("/kaggle/working/rna3d_run/candidates")
    if not base.exists():
        raise FileNotFoundError("No /kaggle/working/rna3d_run/candidates. Run previous stages first.")
    cfg_dirs = sorted([d for d in base.iterdir() if d.is_dir() and d.name.startswith("cfg_")],
                      key=lambda x: x.stat().st_mtime, reverse=True)
    if not cfg_dirs:
        raise FileNotFoundError("No cfg_* found. Run previous stages first.")
    RUN_DIR = cfg_dirs[0]

cfg_cand_path = RUN_DIR / "cfg_candidate_gen.json"
CAND_WRAP = read_json_safe(cfg_cand_path)
CFG_A = CAND_WRAP.get("cfg", {})
splits_default = list(CFG_A.get("RUN_SPLITS", ["val"]))

STD_DIR = pick_latest_dir(RUN_DIR / "standardized", "std_")
STAGEC_DIR = pick_latest_dir(RUN_DIR / "ranking", "rank_")
OPT_DIR = pick_latest_dir(RUN_DIR / "opt", "opt_")  # optional

if not STD_DIR or not STAGEC_DIR:
    raise FileNotFoundError("Missing STD_DIR or STAGEC_DIR. Ensure Stages B and C have been run.")

print("=== STAGE F INPUTS ===")
print("RUN_DIR   :", RUN_DIR)
print("STD_DIR   :", STD_DIR)
print("STAGEC_DIR:", STAGEC_DIR)
print("OPT_DIR   :", OPT_DIR if str(OPT_DIR) else "(none)")

# determine which splits exist
available_topk = []
for sp in ["train", "val", "test"]:
    if (STAGEC_DIR / f"topk_{sp}.parquet").exists():
        available_topk.append(sp)
SELECT_SPLITS = available_topk if available_topk else splits_default
print("SELECT_SPLITS:", SELECT_SPLITS)

# ----------------------------
# 2) pick best bundle; create fallback if missing/incomplete
# ----------------------------
BUNDLE_ROOT = Path("/kaggle/working/rna3d_final_bundle")
safe_mkdir(BUNDLE_ROOT)

def bundle_is_valid(d: Path) -> bool:
    return (d / "feature_cols.json").exists() and (d / "method_categories.json").exists()

def bundle_metric(d: Path) -> float:
    rep = d / "reports" / "train_report.json"
    if rep.exists():
        try:
            j = read_json_safe(rep)
            v = j.get("train_ndcg5_mean", -1e18)
            return float(v) if v is not None else -1e18
        except Exception:
            return -1e18
    return -1e18

bundle_dirs = [d for d in BUNDLE_ROOT.iterdir() if d.is_dir() and d.name.startswith("bundle_")]
valid_bundles = [d for d in bundle_dirs if bundle_is_valid(d)]

if valid_bundles:
    valid_bundles.sort(key=lambda d: (bundle_metric(d), d.stat().st_mtime), reverse=True)
    BUNDLE_DIR = valid_bundles[0]
    print("BEST TRAINED BUNDLE_DIR:", BUNDLE_DIR)
else:
    fb_id = hashlib.sha1(f"fallback|{time.time()}".encode()).hexdigest()[:10]
    BUNDLE_DIR = safe_mkdir(BUNDLE_ROOT / f"bundle_fallback_{fb_id}")
    safe_mkdir(BUNDLE_DIR / "models")
    safe_mkdir(BUNDLE_DIR / "reports")

    # infer methods from Stage C
    methods = []
    for sp in SELECT_SPLITS:
        p = STAGEC_DIR / f"topk_{sp}.parquet"
        if p.exists():
            dfm = pd.read_parquet(p, columns=["method"] if "method" in pd.read_parquet(p).columns else None)
            if "method" in dfm.columns:
                methods.extend(dfm["method"].astype("string").fillna("UNK").unique().tolist())
    methods = sorted(list(dict.fromkeys(methods))) if methods else ["TBM_REF", "TBM", "DRFOLD2", "EXT", "UNK"]

    base_feats = [
        "rank_in",
        "stageC_score","stageC_bond","stageC_curv","stageC_clash","stageC_rg",
        "L","bond_mean","bond_std","bond_p01","bond_p99","curv_mean","rg","clash_rate","coord_absmax",
        "opt_best_energy","opt_best_bond","opt_best_smooth","opt_best_clash",
    ]
    feature_cols = base_feats + [f"method__{m}" for m in methods]
    feature_cols = unique_preserve(feature_cols)

    (BUNDLE_DIR / "feature_cols.json").write_text(json.dumps(feature_cols, indent=2), encoding="utf-8")
    (BUNDLE_DIR / "method_categories.json").write_text(json.dumps(methods, indent=2), encoding="utf-8")
    (BUNDLE_DIR / "feature_fill_stats.json").write_text(json.dumps({}, indent=2), encoding="utf-8")
    (BUNDLE_DIR / "train_cfg.json").write_text(json.dumps({
        "fallback_bundle": True,
        "utc_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
        "note": "No trained ranker available; selection will use deterministic fallback rule."
    }, indent=2), encoding="utf-8")
    (BUNDLE_DIR / "reports" / "train_report.json").write_text(json.dumps({
        "backend": "none",
        "train_ndcg5_mean": None,
        "note": "Training bundle missing; created fallback bundle."
    }, indent=2), encoding="utf-8")
    print("NO VALID TRAINED BUNDLE FOUND -> CREATED FALLBACK BUNDLE:", BUNDLE_DIR)

# load bundle assets (guaranteed)
feature_cols = json.loads((BUNDLE_DIR / "feature_cols.json").read_text())
method_cats  = json.loads((BUNDLE_DIR / "method_categories.json").read_text())
feature_cols = unique_preserve([str(c) for c in feature_cols])  # CRITICAL: remove duplicates
try:
    fill_stats = json.loads((BUNDLE_DIR / "feature_fill_stats.json").read_text())
except Exception:
    fill_stats = {}

# ----------------------------
# 3) load ranker if exists (optional)
# ----------------------------
ranker_backend = None
ranker = None
lgb_path = BUNDLE_DIR / "models" / "ranker_model_lgb.txt"
xgb_path = BUNDLE_DIR / "models" / "ranker_model_xgb.json"

if lgb_path.exists():
    try:
        import lightgbm as lgb
        ranker = lgb.Booster(model_file=str(lgb_path))
        ranker_backend = "lightgbm"
    except Exception as e:
        print("[WARN] Failed to load LightGBM model:", e)

if ranker is None and xgb_path.exists():
    try:
        import xgboost as xgb
        ranker = xgb.Booster()
        ranker.load_model(str(xgb_path))
        ranker_backend = "xgboost"
    except Exception as e:
        print("[WARN] Failed to load XGBoost model:", e)

print("RANKER_BACKEND:", ranker_backend if ranker is not None else "(none -> fallback)")

# ----------------------------
# 4) geometry features (same family as Stage E)
# ----------------------------
try:
    import torch
    _HAS_TORCH = True
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.set_grad_enabled(False)
except Exception:
    _HAS_TORCH = False
    device = None

FEAT_CFG = {"PAIR_SAMPLES": 20000, "FULL_CDIST_MAX_L": 512, "MIN_SEQ_SEP": 4, "CLASH_THRESH": 2.0}

def geom_features(coords: np.ndarray, seed: int) -> Dict[str, float]:
    c = np.asarray(coords)
    L = int(c.shape[0]) if c.ndim == 2 else 0
    out = {
        "L": float(L),
        "bond_mean": np.nan, "bond_std": np.nan, "bond_p01": np.nan, "bond_p99": np.nan,
        "curv_mean": np.nan,
        "rg": np.nan,
        "clash_rate": np.nan,
        "coord_absmax": np.nan,
    }
    if c.ndim != 2 or c.shape[1] != 3 or L < 2 or (not np.isfinite(c).all()):
        return out

    c64 = c.astype(np.float64, copy=False)
    out["coord_absmax"] = float(np.max(np.abs(c64)))

    d = np.linalg.norm(c64[1:] - c64[:-1], axis=1)
    out["bond_mean"] = float(np.mean(d))
    out["bond_std"]  = float(np.std(d))
    out["bond_p01"]  = float(np.quantile(d, 0.01))
    out["bond_p99"]  = float(np.quantile(d, 0.99))

    if L >= 3:
        dd = c64[2:] - 2.0*c64[1:-1] + c64[:-2]
        out["curv_mean"] = float(np.mean(np.sum(dd*dd, axis=1)))
    else:
        out["curv_mean"] = 0.0

    cen = np.mean(c64, axis=0, keepdims=True)
    out["rg"] = float(np.sqrt(np.mean(np.sum((c64 - cen)**2, axis=1)) + 1e-12))

    thr = float(FEAT_CFG["CLASH_THRESH"])
    min_sep = int(FEAT_CFG["MIN_SEQ_SEP"])

    if _HAS_TORCH and L <= int(FEAT_CFG["FULL_CDIST_MAX_L"]):
        tc = torch.as_tensor(c64, dtype=torch.float64, device=device)
        dist = torch.cdist(tc, tc)
        idx = torch.arange(L, device=device)
        sep = (idx[:, None] - idx[None, :]).abs()
        mask = (sep >= min_sep)
        dij = dist[mask]
        out["clash_rate"] = float((dij < thr).double().mean().detach().cpu().numpy()) if dij.numel() else 0.0
    else:
        rng = np.random.default_rng(seed)
        M = int(FEAT_CFG["PAIR_SAMPLES"])
        if L <= min_sep + 1:
            out["clash_rate"] = 0.0
        else:
            i = rng.integers(0, L - min_sep, size=M)
            off = rng.integers(min_sep, L, size=M)
            j = (i + off) % L
            dij = np.linalg.norm(c64[i] - c64[j], axis=1)
            out["clash_rate"] = float(np.mean(dij < thr))

    return out

# ----------------------------
# 5) optimized BEST manifest (optional)
# ----------------------------
def load_opt_best(split: str) -> Optional[pd.DataFrame]:
    if not str(OPT_DIR):
        return None
    p = OPT_DIR / f"opt_manifest_{split}.parquet"
    if not p.exists():
        return None
    df = pd.read_parquet(p)
    if "status" not in df.columns:
        return None
    df = df[df["status"].astype("string") == "BEST"].copy()
    if len(df) == 0:
        return None
    keep = ["target_id","cand_id","opt_coords_path","best_energy","best_bond","best_smooth","best_clash"]
    cols = [c for c in keep if c in df.columns]
    df = df[cols].copy()
    df = df.rename(columns={"opt_coords_path":"best_coords_path"})
    for c in ["target_id","cand_id","best_coords_path"]:
        if c in df.columns:
            df[c] = df[c].astype("string")
    return df

# ----------------------------
# 6) build features from Stage C topk (WITH DUP-COL GUARDS)
# ----------------------------
def build_features_from_topk(split: str) -> pd.DataFrame:
    topk_p = STAGEC_DIR / f"topk_{split}.parquet"
    df = pd.read_parquet(topk_p)

    # hard guard: drop duplicate column names NOW (critical for your error)
    if df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()].copy()

    # ensure base cols
    for c in ["target_id","cand_id","method"]:
        if c not in df.columns:
            df[c] = "UNK"
        df[c] = df[c].astype("string")

    if "rank_in" in df.columns:
        df["rank_in"] = pd.to_numeric(df["rank_in"], errors="coerce").fillna(999).astype("int32")
    elif "rank" in df.columns:
        df["rank_in"] = pd.to_numeric(df["rank"], errors="coerce").fillna(999).astype("int32")
    else:
        df["rank_in"] = 999

    if "std_coords_path" not in df.columns:
        df["std_coords_path"] = ""
    df["std_coords_path"] = df["std_coords_path"].astype("string")

    # join opt best
    opt_best = load_opt_best(split)
    if opt_best is not None:
        df = df.merge(opt_best, on=["target_id","cand_id"], how="left")
    else:
        df["best_coords_path"] = ""
        for c in ["best_energy","best_bond","best_smooth","best_clash"]:
            df[c] = np.nan

    df["coords_path_used"] = df["best_coords_path"].astype("string")
    m = df["coords_path_used"].str.len() == 0
    df.loc[m, "coords_path_used"] = df.loc[m, "std_coords_path"].astype("string")

    # normalize Stage C score fields (if present)
    if "score" in df.columns and "stageC_score" not in df.columns:
        df["stageC_score"] = pd.to_numeric(df["score"], errors="coerce")
    if "score_bond" in df.columns and "stageC_bond" not in df.columns:
        df["stageC_bond"] = pd.to_numeric(df["score_bond"], errors="coerce")
    if "score_curv" in df.columns and "stageC_curv" not in df.columns:
        df["stageC_curv"] = pd.to_numeric(df["score_curv"], errors="coerce")
    if "score_clash" in df.columns and "stageC_clash" not in df.columns:
        df["stageC_clash"] = pd.to_numeric(df["score_clash"], errors="coerce")
    if "score_rg" in df.columns and "stageC_rg" not in df.columns:
        df["stageC_rg"] = pd.to_numeric(df["score_rg"], errors="coerce")

    # normalize opt fields
    if "best_energy" in df.columns and "opt_best_energy" not in df.columns:
        df["opt_best_energy"] = pd.to_numeric(df["best_energy"], errors="coerce")
    if "best_bond" in df.columns and "opt_best_bond" not in df.columns:
        df["opt_best_bond"] = pd.to_numeric(df["best_bond"], errors="coerce")
    if "best_smooth" in df.columns and "opt_best_smooth" not in df.columns:
        df["opt_best_smooth"] = pd.to_numeric(df["best_smooth"], errors="coerce")
    if "best_clash" in df.columns and "opt_best_clash" not in df.columns:
        df["opt_best_clash"] = pd.to_numeric(df["best_clash"], errors="coerce")

    # geometry features
    feats = []
    has_coords = []
    for r in df.itertuples(index=False):
        tid = str(getattr(r, "target_id"))
        cid = str(getattr(r, "cand_id"))
        p = str(getattr(r, "coords_path_used"))
        coords = None
        if p and Path(p).exists():
            coords = load_npz_coords(Path(p))
        ok = coords is not None
        has_coords.append(ok)
        if ok:
            seed = stable_hash_int(f"{split}|{tid}|{cid}")
            feats.append(geom_features(coords, seed=seed))
        else:
            feats.append({"L": np.nan, "bond_mean": np.nan, "bond_std": np.nan, "bond_p01": np.nan, "bond_p99": np.nan,
                          "curv_mean": np.nan, "rg": np.nan, "clash_rate": np.nan, "coord_absmax": np.nan})
    df = pd.concat([df.reset_index(drop=True), pd.DataFrame(feats)], axis=1)
    df["has_coords"] = np.array(has_coords, dtype=bool)

    # method OHE (ensure no duplicate columns)
    for mcat in method_cats:
        col = f"method__{mcat}"
        if col not in df.columns:
            df[col] = (df["method"] == mcat).astype("int32")

    # after all adds: drop duplicate column names AGAIN (critical)
    if df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()].copy()

    # ensure all feature cols exist (no duplicates in feature_cols)
    for c in feature_cols:
        if c not in df.columns:
            df[c] = 0.0

    # Build X by reindex (guarantees 1D per col; no df[col] ambiguity)
    X = df.reindex(columns=feature_cols).copy()

    # Fill NaNs robustly
    X = X.replace([np.inf, -np.inf], np.nan)
    for c in X.columns:
        s = X[c]
        # if still not Series (shouldn't happen), force first column
        if not isinstance(s, pd.Series):
            s = pd.Series(np.asarray(s)[:, 0], index=X.index)
        if pd.api.types.is_bool_dtype(s) or pd.api.types.is_integer_dtype(s):
            X[c] = s.fillna(0)
        else:
            med = float(fill_stats.get(c, 0.0))
            X[c] = pd.to_numeric(s, errors="coerce").fillna(med).astype(np.float64)

    # predict
    if ranker is not None:
        if ranker_backend == "lightgbm":
            df["ranker_score"] = ranker.predict(X)
        else:
            import xgboost as xgb
            df["ranker_score"] = ranker.predict(xgb.DMatrix(X))
    else:
        df["ranker_score"] = np.nan

    return df

# ----------------------------
# 7) select Top-5 per target (pad)
# ----------------------------
def select_top5(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for tid, sub in df.groupby("target_id", sort=False):
        sub = sub.copy()

        if sub["ranker_score"].notna().any():
            sub = sub.sort_values(["ranker_score","rank_in","cand_id"], ascending=[False, True, True])
        else:
            sub["method_pri"] = sub["method"].map(method_priority).astype("int32")
            sub = sub.sort_values(["method_pri","rank_in","cand_id"], ascending=[True, True, True])

        sub["final_rank"] = np.arange(1, len(sub) + 1, dtype=np.int32)
        top = sub.head(5).copy()
        top["is_padded"] = 0

        if len(top) < 5 and len(top) > 0:
            best = top.iloc[[0]].copy()
            needed = 5 - len(top)
            pads = []
            for k in range(needed):
                b = best.copy()
                b["final_rank"] = len(top) + k + 1
                b["is_padded"] = 1
                pads.append(b)
            top = pd.concat([top] + pads, ignore_index=True)

        rows.append(top)

    return pd.concat(rows, ignore_index=True) if rows else df.head(0).copy()

# ----------------------------
# 8) run selection + export
# ----------------------------
FINAL_ID = hashlib.sha1(str(BUNDLE_DIR).encode()).hexdigest()[:10]
FINAL_DIR = safe_mkdir(RUN_DIR / "final_selection" / f"final_{FINAL_ID}")

report = {
    "utc_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "run_dir": str(RUN_DIR),
    "stagec_dir": str(STAGEC_DIR),
    "opt_dir": str(OPT_DIR) if str(OPT_DIR) else "",
    "bundle_dir": str(BUNDLE_DIR),
    "ranker_backend": ranker_backend if ranker is not None else "none_fallback",
    "splits": {},
}

for sp in SELECT_SPLITS:
    topk_p = STAGEC_DIR / f"topk_{sp}.parquet"
    if not topk_p.exists():
        continue

    df_feat = build_features_from_topk(sp)
    df_top5 = select_top5(df_feat)

    outp = FINAL_DIR / f"final_top5_{sp}.parquet"
    df_top5.to_parquet(outp, index=False)

    outcsv = FINAL_DIR / f"final_top5_{sp}.csv"
    keep_cols = ["target_id","final_rank","cand_id","method","rank_in","ranker_score","coords_path_used","has_coords","is_padded"]
    keep_cols = [c for c in keep_cols if c in df_top5.columns]
    df_top5[keep_cols].to_csv(outcsv, index=False)

    report["splits"][sp] = {
        "candidates_in": int(len(df_feat)),
        "targets": int(df_feat["target_id"].nunique()),
        "top5_rows": int(len(df_top5)),
        "with_coords_in": int(df_feat["has_coords"].sum()),
        "with_coords_top5": int(df_top5["has_coords"].sum()) if "has_coords" in df_top5.columns else 0,
        "padded_rows": int(df_top5["is_padded"].sum()) if "is_padded" in df_top5.columns else 0,
        "saved_parquet": str(outp),
        "saved_csv": str(outcsv),
    }

    print(f"[OK] {sp}: saved {outp}")
    print("     ", report["splits"][sp])

rep_path = FINAL_DIR / "final_selection_report.json"
rep_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("\n[OK] saved report:", rep_path)

# ----------------------------
# 9) save best model bundle (copy + zip)
# ----------------------------
BUNDLE_OUT = safe_mkdir(RUN_DIR / "model_bundle_best")

# clean
for p in list(BUNDLE_OUT.glob("*")):
    try:
        if p.is_file():
            p.unlink()
        else:
            shutil.rmtree(p)
    except Exception:
        pass

# copy essential
copy_if_exists(BUNDLE_DIR / "feature_cols.json", BUNDLE_OUT / "feature_cols.json")
copy_if_exists(BUNDLE_DIR / "method_categories.json", BUNDLE_OUT / "method_categories.json")
copy_if_exists(BUNDLE_DIR / "feature_fill_stats.json", BUNDLE_OUT / "feature_fill_stats.json")
copy_if_exists(BUNDLE_DIR / "train_cfg.json", BUNDLE_OUT / "train_cfg.json")
copy_if_exists(BUNDLE_DIR / "reports" / "train_report.json", BUNDLE_OUT / "reports" / "train_report.json")

copy_if_exists(lgb_path, BUNDLE_OUT / "models" / lgb_path.name)
copy_if_exists(xgb_path, BUNDLE_OUT / "models" / xgb_path.name)

copy_if_exists(RUN_DIR / "cfg_candidate_gen.json", BUNDLE_OUT / "cfg_candidate_gen.json")
copy_if_exists(STD_DIR / "cfg_standardize.json", BUNDLE_OUT / "cfg_standardize.json")
copy_if_exists(STAGEC_DIR / "cfg_stageC.json", BUNDLE_OUT / "cfg_stageC.json")
if str(OPT_DIR):
    copy_if_exists(OPT_DIR / "cfg_stageD_opt.json", BUNDLE_OUT / "cfg_stageD_opt.json")
copy_if_exists(rep_path, BUNDLE_OUT / "final_selection_report.json")

# manifest
includes = []
for fp in BUNDLE_OUT.rglob("*"):
    if fp.is_file():
        try:
            includes.append(str(fp.relative_to(BUNDLE_OUT)))
        except Exception:
            pass

manifest = {
    "bundle_created_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "bundle_out": str(BUNDLE_OUT),
    "ranker_backend": report["ranker_backend"],
    "includes": sorted(includes),
}
(BUNDLE_OUT / "bundle_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

zip_path = RUN_DIR / "model_bundle_best.zip"
try:
    if zip_path.exists():
        zip_path.unlink()
except Exception:
    pass

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for fp in BUNDLE_OUT.rglob("*"):
        if fp.is_file():
            z.write(fp, arcname=str(fp.relative_to(BUNDLE_OUT)))

print("\n[OK] MODEL BUNDLE SAVED")
print("BUNDLE_OUT:", BUNDLE_OUT)
print("ZIP      :", zip_path)


=== STAGE F INPUTS ===
RUN_DIR   : /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac
STD_DIR   : /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/standardized/std_3f9797574f
STAGEC_DIR: /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/ranking/rank_551d139223
OPT_DIR   : /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/opt/opt_1b6cd751e419
SELECT_SPLITS: ['val']
NO VALID TRAINED BUNDLE FOUND -> CREATED FALLBACK BUNDLE: /kaggle/working/rna3d_final_bundle/bundle_fallback_b4d7d51adc
RANKER_BACKEND: (none -> fallback)
[OK] val: saved /kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/final_selection/final_16ad34cba9/final_top5_val.parquet
      {'candidates_in': 48, 'targets': 28, 'top5_rows': 140, 'with_coords_in': 0, 'with_coords_top5': 0, 'padded_rows': 92, 'saved_parquet': '/kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/final_selection/final_16ad34cba9/final_top5_val.parquet', 'saved_csv': '/kaggle/working/rna3d_run/candidates/cfg_806e60f69bac/final_selectio